In [1]:
# 1. Mount Drive — every session, first cell
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, shutil, json

# Create a clean local folder structure
repo_dir = '/content/HinglishDB_repo'
os.makedirs(f'{repo_dir}/data/hinglish_splits', exist_ok=True)
os.makedirs(f'{repo_dir}/data/hinglish_finetuning_ready', exist_ok=True)
os.makedirs(f'{repo_dir}/data/hinglish_finetuning_serialized', exist_ok=True)
os.makedirs(f'{repo_dir}/results', exist_ok=True)
os.makedirs(f'{repo_dir}/notebooks', exist_ok=True)
os.makedirs(f'{repo_dir}/src', exist_ok=True)

base = '/content/drive/MyDrive/HindiDB'

# Copy dataset splits
for split in ['train', 'dev', 'test']:
    shutil.copy(
        f'{base}/data/hinglish_splits/{split}_hinglish.json',
        f'{repo_dir}/data/hinglish_splits/{split}_hinglish.json'
    )
    shutil.copy(
        f'{base}/data/hinglish_finetuning_ready/{split}.json',
        f'{repo_dir}/data/hinglish_finetuning_ready/{split}.json'
    )
    shutil.copy(
        f'{base}/data/hinglish_finetuning_serialized/{split}.json',
        f'{repo_dir}/data/hinglish_finetuning_serialized/{split}.json'
    )

# Copy results
shutil.copy(
    f'{base}/results/FINAL_ALL_RESULTS.json',
    f'{repo_dir}/results/FINAL_ALL_RESULTS.json'
)

print("✅ Files collected")
print("\nFolder structure:")
for root, dirs, files in os.walk(repo_dir):
    level = root.replace(repo_dir, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for file in files:
        print(f'{indent}  {file}')

✅ Files collected

Folder structure:
HinglishDB_repo/
  src/
  data/
    hinglish_splits/
      dev_hinglish.json
      test_hinglish.json
      train_hinglish.json
    hinglish_finetuning_ready/
      dev.json
      train.json
      test.json
    hinglish_finetuning_serialized/
      dev.json
      train.json
      test.json
  results/
    FINAL_ALL_RESULTS.json
  notebooks/


In [3]:
# ── translate.py ──────────────────────────────────────────────
translate_code = '''"""
HinglishDB — Translation Pipeline
Translates BIRD English questions to Hinglish
using Llama-3.1-8B-Instant via Groq API.
"""

import json, os, time
from groq import Groq

def translate_to_hinglish(question: str, client: Groq, max_retries: int = 5) -> str:
    """Translates a single English question to Hinglish."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "Translate English database questions to Hinglish "
                            "(Roman script only, no Devanagari). Use natural Hindi "
                            "grammar with English technical terms. Return translation only.\\n"
                            "Examples:\\n"
                            "EN: How many students scored above 90? "
                            "-> HI: Kitne students ne 90 se zyada score kiya?\\n"
                            "EN: List employees ordered by salary. "
                            "-> HI: Employees ko salary ke order mein list karo."
                        )
                    },
                    {"role": "user", "content": question}
                ],
                temperature=0.3,
                max_tokens=200
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            wait = (2 ** attempt) + 1
            if attempt < max_retries - 1:
                print(f"  Attempt {attempt+1} failed: {e} — retrying in {wait}s")
                time.sleep(wait)
            else:
                print(f"  All retries failed for: {question[:50]}")
                return ""

def translate_split(data: list, split_name: str, output_dir: str,
                    client: Groq, save_every: int = 50) -> list:
    """Translates a full split with checkpointing."""
    os.makedirs(output_dir, exist_ok=True)
    checkpoint_path = f"{output_dir}/{split_name}_checkpoint.json"
    final_path = f"{output_dir}/{split_name}_hinglish.json"

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, encoding="utf-8") as f:
            results = json.load(f)
        start_idx = len(results)
        print(f"Resuming {split_name} from index {start_idx}/{len(data)}")
    else:
        results, start_idx = [], 0

    for i in range(start_idx, len(data)):
        row = data[i]
        hinglish_q = translate_to_hinglish(row["question"], client)
        result = dict(row)
        result["hinglish_question"] = hinglish_q
        result["question_en"] = row["question"]
        results.append(result)

        if (i - start_idx + 1) % 25 == 0:
            print(f"  {i+1}/{len(data)} translated")
        if (i - start_idx + 1) % save_every == 0:
            with open(checkpoint_path, "w", encoding="utf-8") as f:
                json.dump(results, f, ensure_ascii=False, indent=2)

        time.sleep(0.1)

    with open(final_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)

    print(f"✅ {split_name}: {len(results)} pairs saved to {final_path}")
    return results


if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--input_dir", required=True,
                        help="Directory containing train_en.json, dev_en.json, test_en.json")
    parser.add_argument("--output_dir", required=True,
                        help="Directory to save translated files")
    parser.add_argument("--api_key", default=None,
                        help="Groq API key (or set GROQ_API_KEY env variable)")
    args = parser.parse_args()

    api_key = args.api_key or os.environ.get("GROQ_API_KEY")
    if not api_key:
        raise ValueError("Provide --api_key or set GROQ_API_KEY environment variable")

    client = Groq(api_key=api_key)

    for split in ["train", "dev", "test"]:
        path = f"{args.input_dir}/{split}_en.json"
        if not os.path.exists(path):
            print(f"Skipping {split} — file not found at {path}")
            continue
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        translate_split(data, split, args.output_dir, client)
'''

with open(f'{repo_dir}/src/translate.py', 'w') as f:
    f.write(translate_code)
print("✅ translate.py written")

✅ translate.py written


In [4]:
# ── train.py ──────────────────────────────────────────────────
train_code = '''"""
HinglishDB — QLoRA Fine-tuning Script
Fine-tunes Qwen-2.5-Coder-3B on HinglishDB using
parameter-efficient QLoRA adapters.
"""

import json, os, argparse
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer


def load_dataset(paths: list, tokenizer) -> Dataset:
    """Loads and combines one or more JSON finetuning files."""
    combined = []
    for path in paths:
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        combined.extend(data)
        print(f"  Loaded {len(data)} pairs from {os.path.basename(path)}")

    formatted = []
    for row in combined:
        prompt = row.get("prompt", "")
        completion = row.get("completion", row.get("SQL", ""))
        if not prompt:
            continue
        full_text = prompt + completion + tokenizer.eos_token
        formatted.append({"text": full_text, "prompt": prompt,
                          "completion": completion})
    print(f"  Combined total: {len(formatted)} examples")
    return Dataset.from_list(formatted)


def main(args):
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(args.model_name,
                                              trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # 4-bit quantization config
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )

    # Load model
    model = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    model = prepare_model_for_kbit_training(model)

    # Apply LoRA
    lora_config = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # Load datasets
    train_paths = args.train_files.split(",")
    train_dataset = load_dataset(train_paths, tokenizer)
    dev_dataset   = load_dataset([args.dev_file], tokenizer)

    # Training arguments
    training_args = TrainingArguments(
        output_dir=args.output_dir,
        max_steps=args.max_steps,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=25,
        save_strategy="steps",
        save_steps=25,
        save_total_limit=16,
        load_best_model_at_end=False,
        fp16=True,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        tokenizer=tokenizer,
        dataset_text_field="text",
        max_seq_length=1024,
        packing=False,
    )

    trainer.train()
    print(f"✅ Training complete. Checkpoints saved to {args.output_dir}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", default="Qwen/Qwen2.5-Coder-3B")
    parser.add_argument("--train_files", required=True,
                        help="Comma-separated paths to training JSON files")
    parser.add_argument("--dev_file", required=True,
                        help="Path to dev JSON file (Hinglish, for eval)")
    parser.add_argument("--output_dir", required=True)
    parser.add_argument("--max_steps", type=int, default=400)
    args = parser.parse_args()
    main(args)
'''

with open(f'{repo_dir}/src/train.py', 'w') as f:
    f.write(train_code)
print("✅ train.py written")

✅ train.py written


In [5]:
# ── evaluate.py ───────────────────────────────────────────────
eval_code = '''"""
HinglishDB — Evaluation Script
Computes BLEU, SQAM, and TSED metrics on model predictions.
"""

import json, re, os, argparse
import sqlglot
from sqlglot import exp
from sacrebleu.metrics import BLEU


# ── BLEU ──────────────────────────────────────────────────────
def compute_bleu(predictions: list) -> float:
    bleu = BLEU(tokenize="none")
    return bleu.corpus_score(
        [p["prediction"] for p in predictions],
        [[p["gold"] for p in predictions]]
    ).score


# ── SQAM ──────────────────────────────────────────────────────
def parse_sql_clauses(sql: str) -> dict:
    sql = sql.strip().rstrip(";")
    patterns = {
        "SELECT":   r"SELECT\\s+(.*?)\\s+FROM",
        "FROM":     r"FROM\\s+(.*?)(?=\\s+WHERE|\\s+GROUP BY|\\s+HAVING|\\s+ORDER BY|\\s+LIMIT|$)",
        "WHERE":    r"WHERE\\s+(.*?)(?=\\s+GROUP BY|\\s+HAVING|\\s+ORDER BY|\\s+LIMIT|$)",
        "GROUP BY": r"GROUP BY\\s+(.*?)(?=\\s+HAVING|\\s+ORDER BY|\\s+LIMIT|$)",
        "HAVING":   r"HAVING\\s+(.*?)(?=\\s+ORDER BY|\\s+LIMIT|$)",
        "ORDER BY": r"ORDER BY\\s+(.*?)(?=\\s+LIMIT|$)",
    }
    return {
        clause: (m.group(1).strip() if (m := re.search(p, sql,
                 re.IGNORECASE | re.DOTALL)) else "")
        for clause, p in patterns.items()
    }


def split_where(text: str) -> set:
    if not text:
        return set()
    parts = re.split(r"\\s+(?:AND|OR)\\s+", text, flags=re.IGNORECASE)
    return set(re.sub(r"\\bt\\d+\\.", "", p.strip().lower()) for p in parts)


def split_clause(text: str) -> set:
    if not text:
        return set()
    return set(re.sub(r"`", "", re.sub(r"\\bt\\d+\\.", "", p.strip().lower()))
               for p in text.split(","))


def compute_sqam(pred: str, gold: str) -> float:
    weights = {"SELECT": 0.30, "FROM": 0.20, "WHERE": 0.25,
               "GROUP BY": 0.10, "HAVING": 0.05, "ORDER BY": 0.10}
    pc, gc = parse_sql_clauses(pred), parse_sql_clauses(gold)
    score = 0.0
    for clause, w in weights.items():
        fn = split_where if clause in ("WHERE", "HAVING") else split_clause
        ps, gs = fn(pc[clause]), fn(gc[clause])
        if not gs and not ps:
            cs = 1.0
        elif not gs or not ps:
            cs = 0.0
        else:
            cs = len(ps & gs) / len(ps | gs)
        score += w * cs
    return score


# ── TSED ──────────────────────────────────────────────────────
def ast_nodes(node) -> list:
    nodes = []
    def walk(n, d=0):
        if n is None:
            return
        r = type(n).__name__
        if hasattr(n, "this") and isinstance(n.this, str):
            r += f":{n.this.lower()}"
        nodes.append((d, r))
        for child in n.args.values():
            if isinstance(child, list):
                for c in child:
                    if isinstance(c, exp.Expression):
                        walk(c, d + 1)
            elif isinstance(child, exp.Expression):
                walk(child, d + 1)
    walk(node)
    return nodes


def tree_edit_distance(a: list, b: list) -> int:
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1,
                           dp[i-1][j-1] + cost)
    return dp[m][n]


def compute_tsed(pred: str, gold: str) -> float:
    try:
        pa = sqlglot.parse_one(pred, read="sqlite")
        ga = sqlglot.parse_one(gold, read="sqlite")
    except Exception:
        return 0.0
    pn, gn = ast_nodes(pa), ast_nodes(ga)
    D = tree_edit_distance(pn, gn)
    N = max(len(pn), len(gn))
    return max(0.0, 1.0 - D / N) if N > 0 else 1.0


# ── Main ──────────────────────────────────────────────────────
def evaluate(predictions_path: str) -> dict:
    with open(predictions_path, encoding="utf-8") as f:
        predictions = json.load(f)

    bleu = compute_bleu(predictions)
    sqam_scores = [compute_sqam(p["prediction"], p["gold"])
                   for p in predictions]
    tsed_scores = [compute_tsed(p["prediction"], p["gold"])
                   for p in predictions]

    avg_sqam = sum(sqam_scores) / len(sqam_scores)
    avg_tsed = sum(tsed_scores) / len(tsed_scores)

    results = {
        "num_predictions": len(predictions),
        "bleu":  round(bleu, 2),
        "sqam":  round(avg_sqam, 4),
        "tsed":  round(avg_tsed, 4),
    }

    print("=" * 45)
    print(f"Results: {os.path.basename(predictions_path)}")
    print("=" * 45)
    print(f"  BLEU : {results['bleu']}")
    print(f"  SQAM : {results['sqam']}")
    print(f"  TSED : {results['tsed']}")
    return results


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--predictions", required=True,
                        help="Path to predictions JSON file")
    parser.add_argument("--output", default=None,
                        help="Optional path to save metrics JSON")
    args = parser.parse_args()

    metrics = evaluate(args.predictions)

    if args.output:
        with open(args.output, "w") as f:
            json.dump(metrics, f, indent=2)
        print(f"Metrics saved to {args.output}")
'''

with open(f'{repo_dir}/src/evaluate.py', 'w') as f:
    f.write(eval_code)
print("✅ evaluate.py written")

✅ evaluate.py written


In [6]:
# Final check — print what we have
print("=== Final repo structure ===")
for root, dirs, files in os.walk(repo_dir):
    dirs[:] = [d for d in dirs if d != '__pycache__']
    level = root.replace(repo_dir, '').count(os.sep)
    print('  ' * level + os.path.basename(root) + '/')
    for file in files:
        size = os.path.getsize(os.path.join(root, file)) / 1024
        print(f"{'  ' * (level+1)}{file}  ({size:.1f} KB)")

# Zip for download
shutil.make_archive('/content/drive/MyDrive/HinglishDB_repo',
                    'zip', repo_dir)
size = os.path.getsize('/content/drive/MyDrive/HinglishDB_repo.zip')/1e6
print(f"\n✅ Zipped — {size:.1f} MB")
print("Download HinglishDB_repo.zip from Drive")

=== Final repo structure ===
HinglishDB_repo/
  src/
    evaluate.py  (5.6 KB)
    train.py  (4.0 KB)
    translate.py  (4.3 KB)
  data/
    hinglish_splits/
      dev_hinglish.json  (295.7 KB)
      test_hinglish.json  (213.0 KB)
      train_hinglish.json  (2816.9 KB)
    hinglish_finetuning_ready/
      dev.json  (1149.8 KB)
      train.json  (14048.4 KB)
      test.json  (672.4 KB)
    hinglish_finetuning_serialized/
      dev.json  (575.8 KB)
      train.json  (6326.1 KB)
      test.json  (383.5 KB)
  results/
    FINAL_ALL_RESULTS.json  (2.6 KB)
  notebooks/

✅ Zipped — 1.8 MB
Download HinglishDB_repo.zip from Drive


In [ ]:
# Save final consolidated results file on Drive
import json

final_results = {
    'paper': 'HinglishDB',
    'model': 'Qwen-2.5-Coder-3B',

    'EX1_prompt_design': {
        'create_table': {'bleu': 20.39, 'sqam': 0.319, 'tsed': 0.308, 'ckpt': 'ck-125'},
        'serialized':   {'bleu': 14.14, 'sqam': 0.255, 'tsed': 0.281, 'ckpt': 'ck-25'},
        'winner': 'CREATE TABLE',
        'darija_reference': {
            'create_table': {'bleu': 36.20, 'sqam': 0.483, 'tsed': 0.286},
            'serialized':   {'bleu': 32.50, 'sqam': 0.472, 'tsed': 0.220}
        }
    },

    'EX2_english_vs_hinglish': {
        'english_ck25':   {'bleu': 12.63, 'sqam': 0.241, 'tsed': 0.285},
        'english_ck175':  {'bleu': 22.71, 'sqam': 0.326, 'tsed': 0.405},
        'hinglish_ck125': {'bleu': 20.39, 'sqam': 0.319, 'tsed': 0.308},
        'gap_hinglish_vs_best_english': {
            'bleu': -2.32, 'sqam': -0.007, 'tsed': -0.097
        },
        'note': 'Undertrained English (ck-25) shows Hinglish winning (+7.76 BLEU). Converged English (ck-175) outperforms Hinglish (-2.32 BLEU). Gap much smaller than DarijaDB (-10.1 BLEU) due to Hinglish English token overlap.'
    },

    'EX3_multilingual': {
        'hinglish_only':           {'bleu': 20.39, 'sqam': 0.319, 'tsed': 0.308},
        'hindi_only':              {'bleu': 18.20, 'sqam': 0.276, 'tsed': 0.289},
        'english_only':            {'bleu': 22.71, 'sqam': 0.326, 'tsed': 0.405},
        'hindi_hinglish':          {'bleu': 21.20, 'sqam': 0.319, 'tsed': 0.348},
        'english_hinglish':        {'bleu': 20.91, 'sqam': 0.296, 'tsed': 0.344},
        'english_hindi_hinglish':  {'bleu': 22.88, 'sqam': 0.329, 'tsed': 0.374},
        'best_combination': 'english_hindi_hinglish',
        'key_findings': [
            'English+Hindi+Hinglish best on BLEU/SQAM (mirrors DarijaDB multilingual finding)',
            'English-only best on TSED — structural accuracy declines when adding multilingual data',
            'Hindi+Hinglish > English+Hinglish on BLEU — related languages help more than distant ones',
            'Adding all three gives best BLEU but not TSED — consistent with DarijaDB Table 7 pattern'
        ]
    },

    'indicdb_comparison': {
        'hinglish_ex_drop_frontier_models': -6.87,
        'our_hinglish_vs_english_gap_bleu': -2.32,
        'note': 'Our fine-tuned 3B model shows smaller Hinglish-English gap (-2.32 BLEU) than IndicDB frontier models (-6.87% EX), suggesting fine-tuning specifically closes the Hinglish gap more effectively than prompting larger models'
    }
}

with open('/content/drive/MyDrive/HindiDB/results/FINAL_ALL_RESULTS.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print("✅ All results saved to Drive: FINAL_ALL_RESULTS.json")
print("\n=== Complete Results Summary ===")
print(json.dumps(final_results, indent=2))

✅ All results saved to Drive: FINAL_ALL_RESULTS.json

=== Complete Results Summary ===
{
  "paper": "HinglishDB",
  "model": "Qwen-2.5-Coder-3B",
  "EX1_prompt_design": {
    "create_table": {
      "bleu": 20.39,
      "sqam": 0.319,
      "tsed": 0.308,
      "ckpt": "ck-125"
    },
    "serialized": {
      "bleu": 14.14,
      "sqam": 0.255,
      "tsed": 0.281,
      "ckpt": "ck-25"
    },
    "winner": "CREATE TABLE",
    "darija_reference": {
      "create_table": {
        "bleu": 36.2,
        "sqam": 0.483,
        "tsed": 0.286
      },
      "serialized": {
        "bleu": 32.5,
        "sqam": 0.472,
        "tsed": 0.22
      }
    }
  },
  "EX2_english_vs_hinglish": {
    "english_ck25": {
      "bleu": 12.63,
      "sqam": 0.241,
      "tsed": 0.285
    },
    "english_ck175": {
      "bleu": 22.71,
      "sqam": 0.326,
      "tsed": 0.405
    },
    "hinglish_ck125": {
      "bleu": 20.39,
      "sqam": 0.319,
      "tsed": 0.308
    },
    "gap_hinglish_vs_best_englis

In [ ]:
import shutil, os

# Check checkpoint size first
ckpt_dir = '/content/drive/MyDrive/HindiDB/models/qwen_english_baseline'
checkpoints = sorted([c for c in os.listdir(ckpt_dir) if c.startswith('checkpoint')])
print("Available checkpoints:", checkpoints)

# Check total size
total_size = 0
for ckpt in checkpoints:
    for root, dirs, files in os.walk(f'{ckpt_dir}/{ckpt}'):
        for f in files:
            total_size += os.path.getsize(os.path.join(root, f))
print(f"Total checkpoint size: {total_size/1e6:.1f} MB")

Available checkpoints: ['checkpoint-100', 'checkpoint-125', 'checkpoint-150', 'checkpoint-175', 'checkpoint-200', 'checkpoint-25', 'checkpoint-50', 'checkpoint-75']
Total checkpoint size: 1541.8 MB


In [ ]:
# Package only checkpoint-175 — that's all we need
import shutil, os

pkg_dir = '/content/kaggle_english_baseline'
if os.path.exists(pkg_dir):
    shutil.rmtree(pkg_dir)
os.makedirs(pkg_dir, exist_ok=True)

# Copy only checkpoint-175
shutil.copytree(
    f'{ckpt_dir}/checkpoint-175',
    f'{pkg_dir}/checkpoint-175'
)

print("Copied checkpoint-175")

# Zip it
shutil.make_archive(
    '/content/drive/MyDrive/English_baseline_ck175',
    'zip',
    pkg_dir
)

size = os.path.getsize('/content/drive/MyDrive/English_baseline_ck175.zip') / 1e6
print(f"✅ Zipped — {size:.1f} MB")
print("Download English_baseline_ck175.zip from Drive and upload to Kaggle as a new dataset")

Copied checkpoint-175
✅ Zipped — 168.2 MB
Download English_baseline_ck175.zip from Drive and upload to Kaggle as a new dataset


In [ ]:
import os

results_dir = '/content/drive/MyDrive/HindiDB/results'
os.makedirs(results_dir, exist_ok=True)
print(f"✅ Results folder ready at: {results_dir}")

✅ Results folder ready at: /content/drive/MyDrive/HindiDB/results


In [ ]:
data_path = '/content/drive/MyDrive/HindiDB/data'

print("Breakdown of data/ subfolder:")
for item in os.listdir(data_path):
    item_path = os.path.join(data_path, item)
    if os.path.isdir(item_path):
        size, count = get_folder_size(item_path)
        print(f"  {item}: {size / 1e9:.2f} GB ({count} files)")
    else:
        size = os.path.getsize(item_path)
        print(f"  {item}: {size / 1e6:.2f} MB (file)")

Breakdown of data/ subfolder:
  bird_raw: 43.95 GB (1503 files)
  splits: 0.00 GB (4 files)
  hindi_translated: 0.00 GB (3 files)
  filtered_train.json: 2.74 MB (file)
  final: 0.00 GB (4 files)
  finetuning_ready: 0.02 GB (3 files)
  finetuning_ready_serialized: 0.01 GB (3 files)


In [ ]:
import zipfile
import os

def zip_with_progress(source_dir, output_path):
    all_files = []
    for dirpath, dirnames, filenames in os.walk(source_dir):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            all_files.append(fp)

    total_files = len(all_files)
    print(f"Total files to zip: {total_files}")

    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for i, file_path in enumerate(all_files):
            arcname = os.path.relpath(file_path, source_dir)
            zf.write(file_path, arcname)
            print(f"Progress: {i+1}/{total_files} — {arcname}")

    final_size = os.path.getsize(output_path) / 1e6
    print(f"\n✅ Done! Zip size: {final_size:.1f} MB")

zip_with_progress(
    source_dir='/content/kaggle_upload',
    output_path='/content/drive/MyDrive/HindiDB_for_kaggle.zip'
)

Total files to zip: 31
Progress: 1/31 — hindi_translated/train_hi.json
Progress: 2/31 — hindi_translated/test_hi.json
Progress: 3/31 — hindi_translated/dev_hi.json
Progress: 4/31 — finetuning_ready_serialized/dev.json
Progress: 5/31 — finetuning_ready_serialized/train.json
Progress: 6/31 — finetuning_ready_serialized/test.json
Progress: 7/31 — checkpoint-75/special_tokens_map.json
Progress: 8/31 — checkpoint-75/trainer_state.json
Progress: 9/31 — checkpoint-75/training_args.bin
Progress: 10/31 — checkpoint-75/tokenizer.json
Progress: 11/31 — checkpoint-75/scheduler.pt
Progress: 12/31 — checkpoint-75/optimizer.pt
Progress: 13/31 — checkpoint-75/vocab.json
Progress: 14/31 — checkpoint-75/merges.txt
Progress: 15/31 — checkpoint-75/rng_state.pth
Progress: 16/31 — checkpoint-75/added_tokens.json
Progress: 17/31 — checkpoint-75/tokenizer_config.json
Progress: 18/31 — checkpoint-75/adapter_config.json
Progress: 19/31 — checkpoint-75/README.md
Progress: 20/31 — checkpoint-75/adapter_model.safe

In [ ]:
# Now check the size of just this targeted folder
total_bytes, file_count = get_folder_size(upload_dir)
print(f"\nTotal size for Kaggle upload: {total_bytes / 1e9:.3f} GB")
print(f"Total files: {file_count}")


Total size for Kaggle upload: 0.227 GB
Total files: 31


In [ ]:
# Check what's actually inside /content/drive right now
import os
print(os.listdir('/content/drive'))

['MyDrive']


In [ ]:
import shutil

# Remove the stale mountpoint folder
shutil.rmtree('/content/drive')

# Now mount fresh
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Create a project folder in your Drive
import os
os.makedirs('/content/drive/MyDrive/HindiDB/data/bird_raw', exist_ok=True)

# Download train set directly (this takes 5-10 mins on Colab)
!wget -O /content/drive/MyDrive/HindiDB/data/bird_raw/train.zip \
  "https://bird-bench.oss-cn-beijing.aliyuncs.com/train.zip"

# Download dev set
!wget -O /content/drive/MyDrive/HindiDB/data/bird_raw/dev.zip \
  "https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip"

Mounted at /content/drive
^C
--2026-06-16 17:59:01--  https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip
Resolving bird-bench.oss-cn-beijing.aliyuncs.com (bird-bench.oss-cn-beijing.aliyuncs.com)... 8.141.181.247
Connecting to bird-bench.oss-cn-beijing.aliyuncs.com (bird-bench.oss-cn-beijing.aliyuncs.com)|8.141.181.247|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 346207293 (330M) [application/zip]
Saving to: ‘/content/drive/MyDrive/HindiDB/data/bird_raw/dev.zip’

/content/drive/MyDr 100%[===================>] 330.17M  5.98MB/s    in 52s     

2026-06-16 17:59:55 (6.30 MB/s) - ‘/content/drive/MyDrive/HindiDB/data/bird_raw/dev.zip’ saved [346207293/346207293]



In [ ]:
# Unzip both into your Drive (so they persist after session ends)
!unzip /content/drive/MyDrive/HindiDB/data/bird_raw/train.zip \
  -d /content/drive/MyDrive/HindiDB/data/bird_raw/

!unzip /content/drive/MyDrive/HindiDB/data/bird_raw/dev.zip \
  -d /content/drive/MyDrive/HindiDB/data/bird_raw/

Archive:  /content/drive/MyDrive/HindiDB/data/bird_raw/train.zip
   creating: /content/drive/MyDrive/HindiDB/data/bird_raw/train/
  inflating: /content/drive/MyDrive/HindiDB/data/bird_raw/__MACOSX/._train  
  inflating: /content/drive/MyDrive/HindiDB/data/bird_raw/train/train_databases.zip  
  inflating: /content/drive/MyDrive/HindiDB/data/bird_raw/__MACOSX/train/._train_databases.zip  
  inflating: /content/drive/MyDrive/HindiDB/data/bird_raw/train/train.json  
  inflating: /content/drive/MyDrive/HindiDB/data/bird_raw/__MACOSX/train/._train.json  
  inflating: /content/drive/MyDrive/HindiDB/data/bird_raw/train/train_gold.sql  
  inflating: /content/drive/MyDrive/HindiDB/data/bird_raw/__MACOSX/train/._train_gold.sql  
  inflating: /content/drive/MyDrive/HindiDB/data/bird_raw/train/train_tables.json  
  inflating: /content/drive/MyDrive/HindiDB/data/bird_raw/__MACOSX/train/._train_tables.json  
Archive:  /content/drive/MyDrive/HindiDB/data/bird_raw/dev.zip
  inflating: /content/drive/My

 **Explore and filter to databases with fewer than 10 tables**

In [ ]:
# Check what's actually inside the train folder
import os

bird_raw = '/content/drive/MyDrive/HindiDB/data/bird_raw'

# List everything at the top level
print("=== Top level of bird_raw ===")
for item in os.listdir(bird_raw):
    print(item)

=== Top level of bird_raw ===
train.zip
dev.zip
train
__MACOSX
dev_20240627


In [ ]:
# Check what's inside 'train' and 'dev_20240627'
print("=== Inside train/ ===")
for item in os.listdir(f'{bird_raw}/train'):
    print(item)

print("\n=== Inside dev_20240627/ ===")
for item in os.listdir(f'{bird_raw}/dev_20240627'):
    print(item)

=== Inside train/ ===
train_gold.sql
train_tables.json
train.json
train_databases.zip
train_databases
__MACOSX

=== Inside dev_20240627/ ===
dev_tied_append.json
dev_tables.json
dev_databases.zip
dev.sql
dev.json
dev_databases
__MACOSX


The files are still zipped inside. So unzipping again.

In [ ]:
# Unzip the databases (this will take a few minutes)
!unzip /content/drive/MyDrive/HindiDB/data/bird_raw/train/train_databases.zip \
  -d /content/drive/MyDrive/HindiDB/data/bird_raw/train/

!unzip /content/drive/MyDrive/HindiDB/data/bird_raw/dev_20240627/dev_databases.zip \
  -d /content/drive/MyDrive/HindiDB/data/bird_raw/dev_20240627/

print("Done unzipping!")

Archive:  /content/drive/MyDrive/HindiDB/data/bird_raw/train/train_databases.zip
   creating: /content/drive/MyDrive/HindiDB/data/bird_raw/train/train_databases/
  inflating: /content/drive/MyDrive/HindiDB/data/bird_raw/train/__MACOSX/._train_databases  
   creating: /content/drive/MyDrive/HindiDB/data/bird_raw/train/train_databases/european_football_1/
  inflating: /content/drive/MyDrive/HindiDB/data/bird_raw/train/__MACOSX/train_databases/._european_football_1  
   creating: /content/drive/MyDrive/HindiDB/data/bird_raw/train/train_databases/sales_in_weather/
  inflating: /content/drive/MyDrive/HindiDB/data/bird_raw/train/__MACOSX/train_databases/._sales_in_weather  
   creating: /content/drive/MyDrive/HindiDB/data/bird_raw/train/train_databases/craftbeer/
  inflating: /content/drive/MyDrive/HindiDB/data/bird_raw/train/__MACOSX/train_databases/._craftbeer  
   creating: /content/drive/MyDrive/HindiDB/data/bird_raw/train/train_databases/soccer_2016/
  inflating: /content/drive/MyDrive/

In [ ]:
import os

db_base = '/content/drive/MyDrive/HindiDB/data/bird_raw/train/train_databases'
dev_base = '/content/drive/MyDrive/HindiDB/data/bird_raw/dev_20240627/dev_databases'

# ── Train databases ──────────────────────────────────────────
all_items = os.listdir(db_base)

# Only process actual directories, skip files like train_tables.json and .DS_Store
db_folders = [item for item in sorted(all_items)
              if os.path.isdir(os.path.join(db_base, item))]

print(f"Total items in train_databases/: {len(all_items)}")
print(f"Actual DB folders: {len(db_folders)}")

sqlite_found = []
sqlite_missing = []

for db_name in db_folders:
    db_path = os.path.join(db_base, db_name, f'{db_name}.sqlite')
    if os.path.exists(db_path):
        sqlite_found.append(db_name)
    else:
        folder_contents = os.listdir(os.path.join(db_base, db_name))
        sqlite_missing.append((db_name, folder_contents))

print(f"\n✅ SQLite found:   {len(sqlite_found)}")
print(f"❌ SQLite missing: {len(sqlite_missing)}")

if sqlite_missing:
    print("\n--- Folders where .sqlite is missing (first 10) ---")
    for db_name, contents in sqlite_missing[:10]:
        print(f"  {db_name}/  →  {contents}")

# ── Dev databases ────────────────────────────────────────────
print("\n" + "="*50)
dev_items = os.listdir(dev_base)
dev_folders = [item for item in sorted(dev_items)
               if os.path.isdir(os.path.join(dev_base, item))]

print(f"Total items in dev_databases/: {len(dev_items)}")
print(f"Actual Dev DB folders: {len(dev_folders)}")

dev_sqlite_found = []
for db_name in dev_folders:
    db_path = os.path.join(dev_base, db_name, f'{db_name}.sqlite')
    if os.path.exists(db_path):
        dev_sqlite_found.append(db_name)

print(f"✅ Dev SQLite found: {len(dev_sqlite_found)}")

Total items in train_databases/: 71
Actual DB folders: 69

✅ SQLite found:   69
❌ SQLite missing: 0

Total items in dev_databases/: 12
Actual Dev DB folders: 11
✅ Dev SQLite found: 11


In [ ]:
import json
import sqlite3
import os

db_base = '/content/drive/MyDrive/HindiDB/data/bird_raw/train/train_databases'
train_json_path = '/content/drive/MyDrive/HindiDB/data/bird_raw/train/train.json'

# Load train data
with open(train_json_path) as f:
    train_data = json.load(f)

print(f"Total pairs in train.json: {len(train_data)}")

# Count tables per database — skip files, only process directories
db_folders = [item for item in os.listdir(db_base)
              if os.path.isdir(os.path.join(db_base, item))]

db_table_counts = {}
skipped = []

for db_name in db_folders:
    db_path = os.path.join(db_base, db_name, f'{db_name}.sqlite')
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT COUNT(*) FROM sqlite_master WHERE type='table'")
        count = cursor.fetchone()[0]
        db_table_counts[db_name] = count
        conn.close()
    except Exception as e:
        skipped.append(db_name)
        print(f"Skipped (corrupted): {db_name} — {e}")

print(f"\nDatabases successfully read: {len(db_table_counts)}")
print(f"Databases skipped: {len(skipped)}")
print(f"Table count range: {min(db_table_counts.values())} to {max(db_table_counts.values())}")

# Show distribution so we understand the data
from collections import Counter
buckets = Counter()
for count in db_table_counts.values():
    if count < 5:       buckets['1-4 tables'] += 1
    elif count < 10:    buckets['5-9 tables'] += 1
    elif count < 20:    buckets['10-19 tables'] += 1
    else:               buckets['20+ tables'] += 1

print("\nTable count distribution:")
for bucket, num in sorted(buckets.items()):
    print(f"  {bucket}: {num} databases")

# Filter to databases with fewer than 10 tables
valid_dbs = {db for db, count in db_table_counts.items() if count < 10}
print(f"\nDatabases with < 10 tables: {len(valid_dbs)}")
print("Valid databases:", sorted(valid_dbs))

# Filter NLQ-SQL pairs
filtered_data = [row for row in train_data if row['db_id'] in valid_dbs]
print(f"\nNLQ-SQL pairs after filtering: {len(filtered_data)}")

# Save to Drive
os.makedirs('/content/drive/MyDrive/HindiDB/data', exist_ok=True)
with open('/content/drive/MyDrive/HindiDB/data/filtered_train.json', 'w') as f:
    json.dump(filtered_data, f, indent=2)

print("\n✅ Saved filtered_train.json to Drive")

# Sanity check — print 3 examples
print("\n=== Sample entries ===")
for item in filtered_data[:3]:
    print(f"DB:       {item['db_id']}")
    print(f"Question: {item['question']}")
    print(f"SQL:      {item['SQL'][:80]}...")
    print("---")

Total pairs in train.json: 9428

Databases successfully read: 69
Databases skipped: 0
Table count range: 2 to 66

Table count distribution:
  1-4 tables: 27 databases
  10-19 tables: 10 databases
  20+ tables: 4 databases
  5-9 tables: 28 databases

Databases with < 10 tables: 55
Valid databases: ['address', 'airline', 'app_store', 'authors', 'beer_factory', 'bike_share_1', 'car_retails', 'cars', 'chicago_crime', 'citeseer', 'codebase_comments', 'coinmarketcap', 'college_completion', 'computer_student', 'cookbook', 'craftbeer', 'cs_semester', 'disney', 'donor', 'european_football_1', 'food_inspection', 'food_inspection_2', 'genes', 'human_resources', 'ice_hockey_draft', 'image_and_language', 'language_corpus', 'law_episode', 'legislator', 'mental_health_survey', 'menu', 'movie', 'movie_platform', 'movielens', 'music_platform_2', 'music_tracker', 'regional_sales', 'restaurant', 'retail_complains', 'retail_world', 'retails', 'sales', 'sales_in_weather', 'shakespeare', 'shipping', 'shooti

Train Dev Test Split


In [ ]:
import json
import random
import os

random.seed(42)  # for reproducibility

# Load filtered data (6,141 pairs across 55 databases, filtered to <10 tables per DB)
with open('/content/drive/MyDrive/HindiDB/data/filtered_train.json') as f:
    filtered_data = json.load(f)

# Get all unique databases
all_dbs = list({row['db_id'] for row in filtered_data})
random.shuffle(all_dbs)

print(f"Total pairs:     {len(filtered_data)}")
print(f"Total databases: {len(all_dbs)}")

# ── Split databases (not pairs) to avoid leakage ─────────────
# Paper used: 47 train / 5 dev / 5 test databases
# We have 55 databases so we use: 45 train / 5 dev / 5 test
n_test = 5
n_dev  = 5
n_train = len(all_dbs) - n_test - n_dev

test_dbs  = set(all_dbs[:n_test])
dev_dbs   = set(all_dbs[n_test:n_test + n_dev])
train_dbs = set(all_dbs[n_test + n_dev:])

print(f"\nDatabase split:")
print(f"  Train DBs: {len(train_dbs)}")
print(f"  Dev DBs:   {len(dev_dbs)}  → {sorted(dev_dbs)}")
print(f"  Test DBs:  {len(test_dbs)} → {sorted(test_dbs)}")

# ── Split pairs by database assignment ───────────────────────
train_split = [row for row in filtered_data if row['db_id'] in train_dbs]
dev_split   = [row for row in filtered_data if row['db_id'] in dev_dbs]
test_split  = [row for row in filtered_data if row['db_id'] in test_dbs]

print(f"\nPair split:")
print(f"  Train: {len(train_split)} pairs")
print(f"  Dev:   {len(dev_split)} pairs")
print(f"  Test:  {len(test_split)} pairs")
print(f"  Total: {len(train_split)+len(dev_split)+len(test_split)} pairs")

# ── Verify zero overlap ───────────────────────────────────────
assert train_dbs & dev_dbs == set(),  "❌ Train/Dev overlap!"
assert train_dbs & test_dbs == set(), "❌ Train/Test overlap!"
assert dev_dbs & test_dbs == set(),   "❌ Dev/Test overlap!"
print("\n✅ Zero database overlap confirmed")

# ── Save splits to Drive ──────────────────────────────────────
splits_dir = '/content/drive/MyDrive/HindiDB/data/splits'
os.makedirs(splits_dir, exist_ok=True)

with open(f'{splits_dir}/train_en.json', 'w') as f:
    json.dump(train_split, f, indent=2)

with open(f'{splits_dir}/dev_en.json', 'w') as f:
    json.dump(dev_split, f, indent=2)

with open(f'{splits_dir}/test_en.json', 'w') as f:
    json.dump(test_split, f, indent=2)

# Save the db assignment so we reuse same split later
split_info = {
    'train_dbs': sorted(train_dbs),
    'dev_dbs':   sorted(dev_dbs),
    'test_dbs':  sorted(test_dbs)
}
with open(f'{splits_dir}/split_info.json', 'w') as f:
    json.dump(split_info, f, indent=2)

print("\n✅ Saved to Drive:")
print(f"   {splits_dir}/train_en.json")
print(f"   {splits_dir}/dev_en.json")
print(f"   {splits_dir}/test_en.json")
print(f"   {splits_dir}/split_info.json")

# ── Quick domain preview ──────────────────────────────────────
print("\n=== Test set databases (these model never sees during training) ===")
for db in sorted(test_dbs):
    count = sum(1 for r in filtered_data if r['db_id'] == db)
    print(f"  {db}: {count} pairs")

Total pairs:     6141
Total databases: 55

Database split:
  Train DBs: 45
  Dev DBs:   5  → ['authors', 'chicago_crime', 'citeseer', 'computer_student', 'software_company']
  Test DBs:  5 → ['airline', 'codebase_comments', 'cookbook', 'mental_health_survey', 'movie']

Pair split:
  Train: 5232 pairs
  Dev:   528 pairs
  Test:  381 pairs
  Total: 6141 pairs

✅ Zero database overlap confirmed

✅ Saved to Drive:
   /content/drive/MyDrive/HindiDB/data/splits/train_en.json
   /content/drive/MyDrive/HindiDB/data/splits/dev_en.json
   /content/drive/MyDrive/HindiDB/data/splits/test_en.json
   /content/drive/MyDrive/HindiDB/data/splits/split_info.json

=== Test set databases (these model never sees during training) ===
  airline: 92 pairs
  codebase_comments: 124 pairs
  cookbook: 69 pairs
  mental_health_survey: 50 pairs
  movie: 46 pairs


In [ ]:
import json

with open('/content/drive/MyDrive/HindiDB/data/splits/split_info.json') as f:
    split_info = json.load(f)

with open('/content/drive/MyDrive/HindiDB/data/splits/train_en.json') as f:
    train_data = json.load(f)
with open('/content/drive/MyDrive/HindiDB/data/splits/dev_en.json') as f:
    dev_data = json.load(f)
with open('/content/drive/MyDrive/HindiDB/data/splits/test_en.json') as f:
    test_data = json.load(f)

train_dbs_actual = sorted({row['db_id'] for row in train_data})
dev_dbs_actual = sorted({row['db_id'] for row in dev_data})
test_dbs_actual = sorted({row['db_id'] for row in test_data})

print(f"train_en.json: {len(train_data)} pairs — matches split_info: {train_dbs_actual == sorted(split_info['train_dbs'])}")
print(f"dev_en.json:   {len(dev_data)} pairs — matches split_info: {dev_dbs_actual == sorted(split_info['dev_dbs'])}")
print(f"test_en.json:  {len(test_data)} pairs — matches split_info: {test_dbs_actual == sorted(split_info['test_dbs'])}")
print(f"\nTotal: {len(train_data) + len(dev_data) + len(test_data)} (should be 6141)")

train_en.json: 5232 pairs — matches split_info: True
dev_en.json:   528 pairs — matches split_info: True
test_en.json:  381 pairs — matches split_info: True

Total: 6141 (should be 6141)


Setting up IndicTrans2.0

In [ ]:
!pip install "transformers==4.43.4" -q

In [ ]:
%cd /content/IndicTransToolkit
!pip install --editable . -q
!pip install cython -q
!python setup.py build_ext --inplace
%cd /content

/content/IndicTransToolkit
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for indictranstoolkit (pyproject.toml) ... done
Compiling IndicTransToolkit/processor.pyx because it changed.
[1/1] Cythonizing IndicTransToolkit/processor.pyx
running build_ext
building 'IndicTransToolkit.processor' extension
x86_64-linux-gnu-gcc -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -fPIC -I/usr/include/python3.12 -c IndicTransToolkit/processor.c -o build/temp.linux-x86_64-cpython-312/IndicTransToolkit/processor.o
x86_64-linux-gnu-gcc -shared -Wl,-O1 -Wl,-Bsymbolic-functions -Wl,-Bsymbolic-functions -g -fwrapv -O2 build/temp.linux-x86_64-cpython-312/IndicTransToolkit/processor.o -L/usr/lib/x86_64-linux-gnu -o build/lib.linux-x86_64-cpython

In [ ]:
# 2. Confirm transformers version after restart
import transformers
print(transformers.__version__)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

4.43.4


In [ ]:
# 3. Login
from google.colab import userdata
from huggingface_hub import login
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("✅ Logged in")

✅ Logged in


In [ ]:
# 4. GPU check
import torch
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [ ]:
!git clone https://github.com/VarunGumma/IndicTransToolkit.git /content/IndicTransToolkit

Cloning into '/content/IndicTransToolkit'...
remote: Enumerating objects: 276, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 276 (delta 82), reused 67 (delta 58), pack-reused 155 (from 1)
Receiving objects: 100% (276/276), 4.39 MiB | 19.31 MiB/s, done.
Resolving deltas: 100% (121/121), done.


In [ ]:
!ls /content/IndicTransToolkit

CHANGELOG.md	   LICENSE	pyproject.toml	setup.py
IndicTransToolkit  MANIFEST.in	README.md


In [ ]:
!find /content/IndicTransToolkit -iname "*processor*"

/content/IndicTransToolkit/IndicTransToolkit/processor.pyx
/content/IndicTransToolkit/IndicTransToolkit/processor.cpython-312-x86_64-linux-gnu.so
/content/IndicTransToolkit/IndicTransToolkit/processor.c
/content/IndicTransToolkit/build/lib.linux-x86_64-cpython-312/IndicTransToolkit/processor.cpython-312-x86_64-linux-gnu.so
/content/IndicTransToolkit/build/temp.linux-x86_64-cpython-312/IndicTransToolkit/processor.o


In [ ]:
import importlib.util

so_path = "/content/IndicTransToolkit/IndicTransToolkit/processor.cpython-312-x86_64-linux-gnu.so"
spec = importlib.util.spec_from_file_location("processor", so_path)
processor_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(processor_module)
IndicProcessor = processor_module.IndicProcessor

ip = IndicProcessor(inference=True)
print("✅ IndicProcessor ready")

✅ IndicProcessor ready


Loading the model

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

model_name = "ai4bharat/indictrans2-en-indic-dist-200M"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)

print("✅ Model loaded successfully")

Using device: cuda


modeling_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

✅ Model loaded successfully


Defining the translate branch function

In [ ]:
def translate_batch(sentences, src_lang="eng_Latn", tgt_lang="hin_Deva", batch_size=8):
    translations = []

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i + batch_size]
        batch_processed = ip.preprocess_batch(batch, src_lang=src_lang, tgt_lang=tgt_lang)

        inputs = tokenizer(
            batch_processed,
            truncation=True,
            padding="longest",
            return_tensors="pt",
            max_length=256
        ).to(DEVICE)

        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs,
                max_length=256,
                num_beams=5,
                num_return_sequences=1
            )

        with tokenizer.as_target_tokenizer():
            decoded = tokenizer.batch_decode(
                generated_tokens.detach().cpu().tolist(),
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True
            )

        decoded = ip.postprocess_batch(decoded, lang=tgt_lang)
        translations.extend(decoded)

        if i % 50 == 0:
            print(f"Translated {i}/{len(sentences)}...")

    return translations

In [ ]:
test_sentences = [
    "Who is the author with the most pages across all books?",
    "Name movie titles released in year 1945.",
    "What is the name of the longest movie title?",
    "What is the shortest duration of rainfall"
]

test_translations = translate_batch(test_sentences)

for en, hi in zip(test_sentences, test_translations):
    print(f"EN: {en}")
    print(f"HI: {hi}")
    print("---")

Translated 0/4...
EN: Who is the author with the most pages across all books?
HI: सभी पुस्तकों में सबसे अधिक पृष्ठ वाले लेखक कौन हैं?
---
EN: Name movie titles released in year 1945.
HI: वर्ष 1945 में रिलीज़ हुई फ़िल्मों के नाम।
---
EN: What is the name of the longest movie title?
HI: सबसे लंबे फिल्म शीर्षक का नाम क्या है?
---
EN: What is the shortest duration of rainfall
HI: वर्षा की सबसे कम अवधि क्या है?
---


CHECKPOINTED TRANSLATION FUNCTION


In [ ]:
import os

def translate_split_with_checkpointing(data, split_name, output_dir, batch_size=8, save_every=20):
    checkpoint_path = f'{output_dir}/{split_name}_checkpoint.json'
    final_path = f'{output_dir}/{split_name}_hi.json'

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path) as f:
            results = json.load(f)
        start_idx = len(results)
        print(f"Resuming from {start_idx} already-translated pairs")
    else:
        results = []
        start_idx = 0

    questions = [row['question'] for row in data]
    remaining = questions[start_idx:]

    for i in range(0, len(remaining), batch_size):
        batch = remaining[i:i + batch_size]
        batch_processed = ip.preprocess_batch(batch, src_lang="eng_Latn", tgt_lang="hin_Deva")

        inputs = tokenizer(
            batch_processed, truncation=True, padding="longest",
            return_tensors="pt", max_length=256
        ).to(DEVICE)

        with torch.no_grad():
            generated_tokens = model.generate(**inputs, max_length=256, num_beams=5)

        decoded = tokenizer.batch_decode(
            generated_tokens.detach().cpu().tolist(),
            skip_special_tokens=True, clean_up_tokenization_spaces=True
        )
        decoded = ip.postprocess_batch(decoded, lang="hin_Deva")

        for j, hindi_text in enumerate(decoded):
            row_idx = start_idx + i + j
            row = dict(data[row_idx])
            row['question_hindi'] = hindi_text
            results.append(row)

        completed = start_idx + i + len(batch)
        print(f"Progress: {completed}/{len(data)}")

        if (i // batch_size) % save_every == 0:
            with open(checkpoint_path, 'w') as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
            print(f"  💾 Checkpoint saved at {completed} pairs")

    with open(final_path, 'w') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)

    print(f"✅ {split_name} complete — {len(results)} pairs saved to {final_path}")
    return results

Run on Dev Set

In [ ]:
output_dir = '/content/drive/MyDrive/HindiDB/data/hindi_translated'
os.makedirs(output_dir, exist_ok=True)

print("Files already in output dir:", os.listdir(output_dir))

Files already in output dir: ['dev_hi.json', 'dev_checkpoint.json']


In [ ]:
output_dir = '/content/drive/MyDrive/HindiDB/data/hindi_translated'
os.makedirs(output_dir, exist_ok=True)

with open('/content/drive/MyDrive/HindiDB/data/splits/dev_en.json') as f:
    dev_data = json.load(f)

dev_results = translate_split_with_checkpointing(dev_data, 'dev', output_dir)

Resuming from 8 already-translated pairs
Progress: 16/528
  💾 Checkpoint saved at 16 pairs
Progress: 24/528
Progress: 32/528
Progress: 40/528
Progress: 48/528
Progress: 56/528
Progress: 64/528
Progress: 72/528
Progress: 80/528
Progress: 88/528
Progress: 96/528
Progress: 104/528
Progress: 112/528
Progress: 120/528
Progress: 128/528
Progress: 136/528
Progress: 144/528
Progress: 152/528
Progress: 160/528
Progress: 168/528
Progress: 176/528
  💾 Checkpoint saved at 176 pairs
Progress: 184/528
Progress: 192/528
Progress: 200/528
Progress: 208/528
Progress: 216/528
Progress: 224/528
Progress: 232/528
Progress: 240/528
Progress: 248/528
Progress: 256/528
Progress: 264/528
Progress: 272/528
Progress: 280/528
Progress: 288/528
Progress: 296/528
Progress: 304/528
Progress: 312/528
Progress: 320/528
Progress: 328/528
Progress: 336/528
  💾 Checkpoint saved at 336 pairs
Progress: 344/528
Progress: 352/528
Progress: 360/528
Progress: 368/528
Progress: 376/528
Progress: 384/528
Progress: 392/528
Progr

Checking the translation

In [ ]:
import random
sample = random.sample(dev_results, 15)
for row in sample:
    print(f"EN: {row['question']}")
    print(f"HI: {row['question_hindi']}")
    print("---")

EN: Where is the coordinate (41.66236555, -87.63470194) located? Give the name of the district.
HI: निर्देशांक (41.66236555,-87.63470194) कहाँ स्थित है? जिले का नाम दें।
---
EN: Which year did the "Internet, Multimedia Systems and Applications" conference publish the most papers?
HI: किस वर्ष "इंटरनेट, मल्टीमीडिया सिस्टम एंड एप्लीकेशन" सम्मेलन ने सबसे अधिक शोध पत्र प्रकाशित किए?
---
EN: Among the male customers, how many of them come from a place with over 30,000 inhabitants?
HI: पुरुष ग्राहकों में से कितने 30,000 से अधिक निवासियों वाले स्थान से आते हैं?
---
EN: What is the paper "Stitching videos streamed by mobile phones in real-time" about?
HI: "वास्तविक समय में मोबाइल फोन द्वारा प्रसारित सिलाई वीडियो" किस बारे में है?
---
EN: Who is the author of the paper titled "Open Sourcing Social Solutions (Building Communities of Change)"?
HI: "ओपन सोर्सिंग सोशल सॉल्यूशंस (बिल्डिंग कम्युनिटीज ऑफ चेंज)" शीर्षक वाले पेपर के लेखक कौन हैं?
---
EN: Among all the DB class type citation, which word 

running on test adn train sets


In [ ]:
test_results = translate_split_with_checkpointing(test_data, 'test', output_dir)

Progress: 8/381
  💾 Checkpoint saved at 8 pairs
Progress: 16/381
Progress: 24/381
Progress: 32/381
Progress: 40/381
Progress: 48/381
Progress: 56/381
Progress: 64/381
Progress: 72/381
Progress: 80/381
Progress: 88/381
Progress: 96/381
Progress: 104/381
Progress: 112/381
Progress: 120/381
Progress: 128/381
Progress: 136/381
Progress: 144/381
Progress: 152/381
Progress: 160/381
Progress: 168/381
  💾 Checkpoint saved at 168 pairs
Progress: 176/381
Progress: 184/381
Progress: 192/381
Progress: 200/381
Progress: 208/381
Progress: 216/381
Progress: 224/381
Progress: 232/381
Progress: 240/381
Progress: 248/381
Progress: 256/381
Progress: 264/381
Progress: 272/381
Progress: 280/381
Progress: 288/381
Progress: 296/381
Progress: 304/381
Progress: 312/381
Progress: 320/381
Progress: 328/381
  💾 Checkpoint saved at 328 pairs
Progress: 336/381
Progress: 344/381
Progress: 352/381
Progress: 360/381
Progress: 368/381
Progress: 376/381
Progress: 381/381
✅ test complete — 381 pairs saved to /content/dri

In [ ]:
train_results = translate_split_with_checkpointing(train_data, 'train', output_dir)

Progress: 8/5232
  💾 Checkpoint saved at 8 pairs
Progress: 16/5232
Progress: 24/5232
Progress: 32/5232
Progress: 40/5232
Progress: 48/5232
Progress: 56/5232
Progress: 64/5232
Progress: 72/5232
Progress: 80/5232
Progress: 88/5232
Progress: 96/5232
Progress: 104/5232
Progress: 112/5232
Progress: 120/5232
Progress: 128/5232
Progress: 136/5232
Progress: 144/5232
Progress: 152/5232
Progress: 160/5232
Progress: 168/5232
  💾 Checkpoint saved at 168 pairs
Progress: 176/5232
Progress: 184/5232
Progress: 192/5232
Progress: 200/5232
Progress: 208/5232
Progress: 216/5232
Progress: 224/5232
Progress: 232/5232
Progress: 240/5232
Progress: 248/5232
Progress: 256/5232
Progress: 264/5232
Progress: 272/5232
Progress: 280/5232
Progress: 288/5232
Progress: 296/5232
Progress: 304/5232
Progress: 312/5232
Progress: 320/5232
Progress: 328/5232
  💾 Checkpoint saved at 328 pairs
Progress: 336/5232
Progress: 344/5232
Progress: 352/5232
Progress: 360/5232
Progress: 368/5232
Progress: 376/5232
Progress: 384/5232
P

CONSOLIDATING RESULTS

In [ ]:
import json
import os

translated_dir = '/content/drive/MyDrive/HindiDB/data/hindi_translated'

# Check what's actually in the folder right now
print("Files in hindi_translated/:")
for f in os.listdir(translated_dir):
    print(f"  {f}")

translated_dir = '/content/drive/MyDrive/HindiDB/data/hindi_translated'

splits = {}
for split_name in ['train', 'dev', 'test']:
    with open(f'{translated_dir}/{split_name}_hi.json') as f:
        splits[split_name] = json.load(f)
    print(f"{split_name}: {len(splits[split_name])} pairs")

print(f"\nTotal: {sum(len(v) for v in splits.values())} pairs (should be 6141)")

Files in hindi_translated/:
  dev_hi.json
  test_hi.json
  train_hi.json
train: 5232 pairs
dev: 528 pairs
test: 381 pairs

Total: 6141 pairs (should be 6141)


In [ ]:
# Check for empty, missing, or suspiciously short Hindi translations
for split_name, data in splits.items():
    issues = []
    for i, row in enumerate(data):
        hindi = row.get('question_hindi', '')
        if not hindi or hindi.strip() == '':
            issues.append((i, 'EMPTY', row['question']))
        elif len(hindi.strip()) < 5:
            issues.append((i, 'TOO_SHORT', row['question'], hindi))

    print(f"\n{split_name}: {len(issues)} potential issues out of {len(data)}")
    for issue in issues[:5]:  # show first 5
        print(f"  {issue}")


train: 0 potential issues out of 5232

dev: 0 potential issues out of 528

test: 0 potential issues out of 381


In [ ]:
# Check required fields are all present and intact
required_fields = ['db_id', 'question', 'SQL', 'question_hindi']

for split_name, data in splits.items():
    missing_field_count = 0
    for row in data:
        if not all(field in row for field in required_fields):
            missing_field_count += 1
    print(f"{split_name}: {missing_field_count} rows missing required fields")

train: 0 rows missing required fields
dev: 0 rows missing required fields
test: 0 rows missing required fields


In [ ]:
# Verify zero database overlap is still intact across the Hindi splits
train_dbs = {row['db_id'] for row in splits['train']}
dev_dbs = {row['db_id'] for row in splits['dev']}
test_dbs = {row['db_id'] for row in splits['test']}

print(f"\nTrain DBs: {len(train_dbs)}")
print(f"Dev DBs: {len(dev_dbs)} → {sorted(dev_dbs)}")
print(f"Test DBs: {len(test_dbs)} → {sorted(test_dbs)}")

assert train_dbs & dev_dbs == set(), "❌ Train/Dev overlap!"
assert train_dbs & test_dbs == set(), "❌ Train/Test overlap!"
assert dev_dbs & test_dbs == set(), "❌ Dev/Test overlap!"
print("\n✅ Zero overlap confirmed")


Train DBs: 45
Dev DBs: 5 → ['authors', 'chicago_crime', 'citeseer', 'computer_student', 'software_company']
Test DBs: 5 → ['airline', 'codebase_comments', 'cookbook', 'mental_health_survey', 'movie']

✅ Zero overlap confirmed


FINAL pushing into drive

In [ ]:
import json
import os

translated_dir = '/content/drive/MyDrive/HindiDB/data/hindi_translated'
final_dir = '/content/drive/MyDrive/HindiDB/data/final'
os.makedirs(final_dir, exist_ok=True)

splits = {}
for split_name in ['train', 'dev', 'test']:
    with open(f'{translated_dir}/{split_name}_hi.json') as f:
        splits[split_name] = json.load(f)

# Clean up and standardize field names for the final dataset
# This mirrors Dialect2SQL's structure: db_id, question (Hindi), question_en (English), SQL, evidence
for split_name, data in splits.items():
    final_data = []
    for row in data:
        final_data.append({
            'db_id': row['db_id'],
            'question': row['question_hindi'],      # Hindi is now the primary question
            'question_en': row['question'],          # Keep English for reference/comparison
            'SQL': row['SQL'],
            'evidence': row.get('evidence', '')       # BIRD sometimes has this field
        })

    out_path = f'{final_dir}/{split_name}_hi.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(final_data, f, ensure_ascii=False, indent=2)

    print(f"✅ Saved {split_name}_hi.json — {len(final_data)} pairs")

print(f"\nFinal dataset saved to: {final_dir}")

✅ Saved train_hi.json — 5232 pairs
✅ Saved dev_hi.json — 528 pairs
✅ Saved test_hi.json — 381 pairs

Final dataset saved to: /content/drive/MyDrive/HindiDB/data/final


In [ ]:
# Save a dataset summary/metadata file — useful for your paper's "Dataset" section
metadata = {
    'dataset_name': 'HindiDB',
    'description': 'First text-to-SQL dataset for Hindi, derived from BIRD via IndicTrans2 translation',
    'source_dataset': 'BIRD (filtered to databases with <10 tables)',
    'translation_model': 'ai4bharat/indictrans2-en-indic-dist-200M',
    'splits': {
        'train': len(splits['train']),
        'dev': len(splits['dev']),
        'test': len(splits['test'])
    },
    'total_pairs': sum(len(v) for v in splits.values()),
    'total_databases': 55,
    'train_databases': sorted({row['db_id'] for row in splits['train']}),
    'dev_databases': sorted({row['db_id'] for row in splits['dev']}),
    'test_databases': sorted({row['db_id'] for row in splits['test']})
}

with open(f'{final_dir}/dataset_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("✅ Saved dataset_metadata.json")
print(f"\nSummary:")
print(f"  Train: {metadata['splits']['train']} pairs")
print(f"  Dev:   {metadata['splits']['dev']} pairs")
print(f"  Test:  {metadata['splits']['test']} pairs")
print(f"  Total: {metadata['total_pairs']} pairs across {metadata['total_databases']} databases")

✅ Saved dataset_metadata.json

Summary:
  Train: 5232 pairs
  Dev:   528 pairs
  Test:  381 pairs
  Total: 6141 pairs across 55 databases


In [ ]:
# Final sanity check — print 3 complete examples from the final train file
with open(f'{final_dir}/train_hi.json', encoding='utf-8') as f:
    final_train = json.load(f)

print("=== Sample final entries ===")
for item in final_train[:3]:
    print(f"DB:        {item['db_id']}")
    print(f"Hindi Q:   {item['question']}")
    print(f"English Q: {item['question_en']}")
    print(f"SQL:       {item['SQL'][:80]}...")
    print("---")

=== Sample final entries ===
DB:        movie_platform
Hindi Q:   वर्ष 1945 में रिलीज़ हुई फ़िल्मों के नाम लिखिए। फ़िल्म की लोकप्रियता के अवरोही क्रम के अनुसार सूची को क्रमबद्ध कीजिए।
English Q: Name movie titles released in year 1945. Sort the listing by the descending order of movie popularity.
SQL:       SELECT movie_title FROM movies WHERE movie_release_year = 1945 ORDER BY movie_po...
---
DB:        movie_platform
Hindi Q:   सबसे लोकप्रिय फिल्म के बारे में बताइए? यह कब रिलीज़ हुई थी और फिल्म के निर्देशक कौन हैं?
English Q: State the most popular movie? When was it released and who is the director for the movie?
SQL:       SELECT movie_title, movie_release_year, director_name FROM movies ORDER BY movie...
---
DB:        movie_platform
Hindi Q:   सबसे लंबे फिल्म शीर्षक का नाम क्या है? यह कब रिलीज़ हुई थी?
English Q: What is the name of the longest movie title? When was it released?
SQL:       SELECT movie_title, movie_release_year FROM movies ORDER BY LENGTH(movie_popular...
---


Building the CREATE TABLE prompt builder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sqlite3
import os

db_base = '/content/drive/MyDrive/HindiDB/data/bird_raw/train/train_databases'

def get_schema_ddl(db_id, db_base=db_base):
    """
    Connects to a database's .sqlite file and extracts the original
    CREATE TABLE statements exactly as stored in sqlite_master.
    """
    db_path = os.path.join(db_base, db_id, f'{db_id}.sqlite')

    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database not found: {db_path}")

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT sql FROM sqlite_master WHERE type='table' AND sql IS NOT NULL")
    statements = [row[0] for row in cursor.fetchall()]
    conn.close()

    return ";\n".join(statements) + ";"

In [ ]:
# Test it on a database we know
test_schema = get_schema_ddl('movie_platform')
print(test_schema)

CREATE TABLE "lists"
(
    user_id                     INTEGER
        references lists_users (user_id),
    list_id                     INTEGER not null
        primary key,
    list_title                  TEXT,
    list_movie_number           INTEGER,
    list_update_timestamp_utc   TEXT,
    list_creation_timestamp_utc TEXT,
    list_followers              INTEGER,
    list_url                    TEXT,
    list_comments               INTEGER,
    list_description            TEXT,
    list_cover_image_url        TEXT,
    list_first_image_url        TEXT,
    list_second_image_url       TEXT,
    list_third_image_url        TEXT
);
CREATE TABLE "movies"
(
    movie_id             INTEGER not null
        primary key,
    movie_title          TEXT,
    movie_release_year   INTEGER,
    movie_url            TEXT,
    movie_title_language TEXT,
    movie_popularity     INTEGER,
    movie_image_url      TEXT,
    director_id          TEXT,
    director_name        TEXT,
    director_url 

combining the hindi question

In [ ]:
def build_prompt(db_id, hindi_question, db_base=db_base):
    """
    Builds the full CREATE TABLE-style prompt for fine-tuning,
    mirroring the paper's Listing 1 format.
    """
    schema = get_schema_ddl(db_id, db_base)

    prompt = f"""{schema}

-- Using valid SQLite, answer the following hindi question for the tables provided above.

-- {hindi_question}

"""
    return prompt

In [ ]:
# Test on a real example from your dataset
import json

with open('/content/drive/MyDrive/HindiDB/data/final/train_hi.json', encoding='utf-8') as f:
    train_data = json.load(f)

sample = train_data[0]
test_prompt = build_prompt(sample['db_id'], sample['question'])

print(test_prompt)
print("=" * 50)
print("Target SQL (what the model should learn to generate):")
print(sample['SQL'])

CREATE TABLE "lists"
(
    user_id                     INTEGER
        references lists_users (user_id),
    list_id                     INTEGER not null
        primary key,
    list_title                  TEXT,
    list_movie_number           INTEGER,
    list_update_timestamp_utc   TEXT,
    list_creation_timestamp_utc TEXT,
    list_followers              INTEGER,
    list_url                    TEXT,
    list_comments               INTEGER,
    list_description            TEXT,
    list_cover_image_url        TEXT,
    list_first_image_url        TEXT,
    list_second_image_url       TEXT,
    list_third_image_url        TEXT
);
CREATE TABLE "movies"
(
    movie_id             INTEGER not null
        primary key,
    movie_title          TEXT,
    movie_release_year   INTEGER,
    movie_url            TEXT,
    movie_title_language TEXT,
    movie_popularity     INTEGER,
    movie_image_url      TEXT,
    director_id          TEXT,
    director_name        TEXT,
    director_url 

Applying the prompt builder across the entire datasets

In [ ]:
import json

def assemble_finetuning_dataset(data, db_base=db_base):
    """
    Applies build_prompt to every row, creating (prompt, target_sql) pairs
    ready for fine-tuning.
    """
    finetuning_data = []
    skipped = []

    for row in data:
        try:
            prompt = build_prompt(row['db_id'], row['question'], db_base)
            finetuning_data.append({
                'db_id': row['db_id'],
                'prompt': prompt,
                'completion': row['SQL'],
                'question_en': row.get('question_en', '')  # kept for reference
            })
        except Exception as e:
            skipped.append((row['db_id'], str(e)))

    if skipped:
        print(f"⚠️ Skipped {len(skipped)} rows due to errors:")
        for db_id, err in skipped[:5]:
            print(f"  {db_id}: {err}")

    return finetuning_data

In [ ]:
# Run on all three splits
with open('/content/drive/MyDrive/HindiDB/data/final/train_hi.json', encoding='utf-8') as f:
    train_data = json.load(f)
with open('/content/drive/MyDrive/HindiDB/data/final/dev_hi.json', encoding='utf-8') as f:
    dev_data = json.load(f)
with open('/content/drive/MyDrive/HindiDB/data/final/test_hi.json', encoding='utf-8') as f:
    test_data = json.load(f)

print("Building train prompts...")
train_ft = assemble_finetuning_dataset(train_data)

print("Building dev prompts...")
dev_ft = assemble_finetuning_dataset(dev_data)

print("Building test prompts...")
test_ft = assemble_finetuning_dataset(test_data)

print(f"\nTrain: {len(train_ft)}/{len(train_data)} prompts built")
print(f"Dev:   {len(dev_ft)}/{len(dev_data)} prompts built")
print(f"Test:  {len(test_ft)}/{len(test_data)} prompts built")

Building train prompts...
Building dev prompts...
Building test prompts...

Train: 5232/5232 prompts built
Dev:   528/528 prompts built
Test:  381/381 prompts built


In [ ]:
# Save the fine-tuning-ready dataset
ft_dir = '/content/drive/MyDrive/HindiDB/data/finetuning_ready'
os.makedirs(ft_dir, exist_ok=True)

for split_name, split_data in [('train', train_ft), ('dev', dev_ft), ('test', test_ft)]:
    out_path = f'{ft_dir}/{split_name}.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(split_data, f, ensure_ascii=False, indent=2)
    print(f"✅ Saved {out_path}")

✅ Saved /content/drive/MyDrive/HindiDB/data/finetuning_ready/train.json
✅ Saved /content/drive/MyDrive/HindiDB/data/finetuning_ready/dev.json
✅ Saved /content/drive/MyDrive/HindiDB/data/finetuning_ready/test.json


Fine Tuning


In [ ]:
# Uninstall the broken bitsandbytes and reinstall the correct version
!pip install "bitsandbytes==0.49.2" -q
!pip install "transformers==4.43.4" -q
!pip install "peft==0.11.1" -q
!pip install "accelerate==0.30.0" -q
!pip install "trl==0.8.6" -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 122.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.4/302.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 13.9 MB/s eta 0:00:00


In [ ]:
!pip uninstall bitsandbytes -y -q

# Clear pip cache for bitsandbytes specifically to avoid stale wheel issues
!pip cache remove bitsandbytes

# Fresh install with no cache
!pip install "bitsandbytes==0.49.2" --no-cache-dir -q

Files removed: 0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 56.6 MB/s eta 0:00:00


In [ ]:
import bitsandbytes as bnb
print(f"bitsandbytes: {bnb.__version__}")

bitsandbytes: 0.49.2


In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))
print("✅ HuggingFace login successful")

✅ HuggingFace login successful


In [ ]:
import wandb
wandb.login(key=userdata.get('WANDB_API_KEY'))
print("✅ Weights & Biases login successful")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aniruddha_2006 (aniruddha_2006-sant-longowal-institute-of-engineering-an) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ Weights & Biases login successful


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
model_name = "Qwen/Qwen2.5-Coder-3B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print(f"✅ Model loaded")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/139 [00:00<?, ?B/s]

✅ Model loaded
GPU memory used: 2.66 GB


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for k-bit training — freezes base weights, enables gradient checkpointing
model = prepare_model_for_kbit_training(model)

# LoRA config — standard practice for 3B-scale code models
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


EXPERIMENT 1: Building compact prompt

In [ ]:
import sqlite3
import os
import json

# Redefine db_base — points to where your BIRD .sqlite files live
db_base = '/content/drive/MyDrive/HindiDB/data/bird_raw/train/train_databases'

# Load a sample from your final dataset to test with
with open('/content/drive/MyDrive/HindiDB/data/final/train_hi.json', encoding='utf-8') as f:
    train_data_final = json.load(f)

sample = train_data_final[0]
print(f"DB: {sample['db_id']}")
print(f"Question: {sample['question']}")

DB: movie_platform
Question: वर्ष 1945 में रिलीज़ हुई फ़िल्मों के नाम लिखिए। फ़िल्म की लोकप्रियता के अवरोही क्रम के अनुसार सूची को क्रमबद्ध कीजिए।


In [ ]:
def build_serialized_prompt(db_id, hindi_question, db_base=db_base):
    """
    Builds the compact serialized prompt format from the paper's Listing 2:
    db_name | table1: col1, col2 | table2: col1, col2 ...
    """
    db_path = os.path.join(db_base, db_id, f'{db_id}.sqlite')

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Get all table names
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    table_names = [row[0] for row in cursor.fetchall()]

    table_parts = []
    for table in table_names:
        cursor.execute(f"PRAGMA table_info('{table}')")
        columns = [row[1] for row in cursor.fetchall()]  # row[1] = column name
        table_parts.append(f"{table}: {', '.join(columns)}")

    conn.close()

    schema_str = f"{db_id} | " + " | ".join(table_parts)

    prompt = f"""{schema_str}

-- Using valid SQLite, answer the following hindi question for the tables provided above.

-- {hindi_question}

"""
    return prompt

In [ ]:
test_serialized = build_serialized_prompt(sample['db_id'], sample['question'])
print(test_serialized)

movie_platform | lists: user_id, list_id, list_title, list_movie_number, list_update_timestamp_utc, list_creation_timestamp_utc, list_followers, list_url, list_comments, list_description, list_cover_image_url, list_first_image_url, list_second_image_url, list_third_image_url | movies: movie_id, movie_title, movie_release_year, movie_url, movie_title_language, movie_popularity, movie_image_url, director_id, director_name, director_url | ratings_users: user_id, rating_date_utc, user_trialist, user_subscriber, user_avatar_image_url, user_cover_image_url, user_eligible_for_trial, user_has_payment_method | lists_users: user_id, list_id, list_update_date_utc, list_creation_date_utc, user_trialist, user_subscriber, user_avatar_image_url, user_cover_image_url, user_eligible_for_trial, user_has_payment_method | ratings: movie_id, rating_id, rating_url, rating_score, rating_timestamp_utc, critic, critic_likes, critic_comments, user_id, user_trialist, user_subscriber, user_eligible_for_trial, use

Applying to entire dataset

In [ ]:
def assemble_serialized_dataset(data, db_base=db_base):
    """
    Applies build_serialized_prompt to every row, creating (prompt, target_sql) pairs.
    """
    serialized_data = []
    skipped = []

    for row in data:
        try:
            prompt = build_serialized_prompt(row['db_id'], row['question'], db_base)
            serialized_data.append({
                'db_id': row['db_id'],
                'prompt': prompt,
                'completion': row['SQL'],
                'question_en': row.get('question_en', '')
            })
        except Exception as e:
            skipped.append((row['db_id'], str(e)))

    if skipped:
        print(f"⚠️ Skipped {len(skipped)} rows:")
        for db_id, err in skipped[:5]:
            print(f"  {db_id}: {err}")

    return serialized_data

In [ ]:
# Load all three splits from your final dataset
with open('/content/drive/MyDrive/HindiDB/data/final/train_hi.json', encoding='utf-8') as f:
    train_data_final = json.load(f)
with open('/content/drive/MyDrive/HindiDB/data/final/dev_hi.json', encoding='utf-8') as f:
    dev_data_final = json.load(f)
with open('/content/drive/MyDrive/HindiDB/data/final/test_hi.json', encoding='utf-8') as f:
    test_data_final = json.load(f)

print("Building serialized train prompts...")
train_serialized = assemble_serialized_dataset(train_data_final)

print("Building serialized dev prompts...")
dev_serialized = assemble_serialized_dataset(dev_data_final)

print("Building serialized test prompts...")
test_serialized_full = assemble_serialized_dataset(test_data_final)

print(f"\nTrain: {len(train_serialized)}/{len(train_data_final)}")
print(f"Dev:   {len(dev_serialized)}/{len(dev_data_final)}")
print(f"Test:  {len(test_serialized_full)}/{len(test_data_final)}")

Building serialized train prompts...
Building serialized dev prompts...
Building serialized test prompts...

Train: 5232/5232
Dev:   528/528
Test:  381/381


In [ ]:
# Save the serialized-format dataset (separate from your CREATE TABLE version)
serialized_dir = '/content/drive/MyDrive/HindiDB/data/finetuning_ready_serialized'
os.makedirs(serialized_dir, exist_ok=True)

for split_name, split_data in [('train', train_serialized), ('dev', dev_serialized), ('test', test_serialized_full)]:
    out_path = f'{serialized_dir}/{split_name}.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(split_data, f, ensure_ascii=False, indent=2)
    print(f"✅ Saved {out_path}")

✅ Saved /content/drive/MyDrive/HindiDB/data/finetuning_ready_serialized/train.json
✅ Saved /content/drive/MyDrive/HindiDB/data/finetuning_ready_serialized/dev.json
✅ Saved /content/drive/MyDrive/HindiDB/data/finetuning_ready_serialized/test.json


Starting the fine tuning

In [ ]:
from datasets import Dataset
import json

def load_as_hf_dataset(path):
    """Loads a JSON file and formats it as a HuggingFace Dataset for training."""
    with open(path, encoding='utf-8') as f:
        data = json.load(f)

    # Combine prompt + completion into a single text field for causal LM training
    formatted = []
    for row in data:
        full_text = row['prompt'] + row['completion'] + tokenizer.eos_token
        formatted.append({'text': full_text, 'prompt': row['prompt'], 'completion': row['completion']})

    return Dataset.from_list(formatted)

# Load CREATE TABLE format data for Run A
train_dataset_a = load_as_hf_dataset('/content/drive/MyDrive/HindiDB/data/finetuning_ready/train.json')
dev_dataset_a = load_as_hf_dataset('/content/drive/MyDrive/HindiDB/data/finetuning_ready/dev.json')

print(f"Train (CREATE TABLE): {len(train_dataset_a)} examples")
print(f"Dev (CREATE TABLE):   {len(dev_dataset_a)} examples")
print(f"\nSample text length: {len(train_dataset_a[0]['text'])} characters")

Train (CREATE TABLE): 5232 examples
Dev (CREATE TABLE):   528 examples

Sample text length: 2980 characters


In [ ]:
# Check token length distribution — important for setting max_seq_length correctly
sample_lengths = [len(tokenizer(ex['text'])['input_ids']) for ex in train_dataset_a.select(range(100))]
print(f"Token length — min: {min(sample_lengths)}, max: {max(sample_lengths)}, avg: {sum(sample_lengths)/len(sample_lengths):.0f}")

Token length — min: 590, max: 836, avg: 689


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/HindiDB/models/qwen_ex1_createtable',
    num_train_epochs=3,
    per_device_train_batch_size=1,        # reduced from 4
    per_device_eval_batch_size=1,         # reduced from 4
    gradient_accumulation_steps=16,        # increased to compensate — same effective batch size of 16
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    report_to="wandb",
    run_name="qwen-ex1-createtable",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_a,
    eval_dataset=dev_dataset_a,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=1024,
    packing=False,
)

print("✅ Trainer re-initialized with reduced batch size")

Map:   0%|          | 0/5232 [00:00<?, ? examples/s]

Map:   0%|          | 0/528 [00:00<?, ? examples/s]

✅ Trainer re-initialized with reduced batch size


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:
import os
checkpoint_dir = '/content/drive/MyDrive/HindiDB/models/qwen_ex1_createtable'
print(os.listdir(checkpoint_dir))

['checkpoint-400', 'checkpoint-500']


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-3B", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-Coder-3B",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
print("✅ Fresh model + LoRA loaded")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✅ Fresh model + LoRA loaded


In [ ]:
import json
from datasets import Dataset

def load_as_hf_dataset(path):
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    formatted = []
    for row in data:
        full_text = row['prompt'] + row['completion'] + tokenizer.eos_token
        formatted.append({'text': full_text, 'prompt': row['prompt'], 'completion': row['completion']})
    return Dataset.from_list(formatted)

train_dataset_a = load_as_hf_dataset('/content/drive/MyDrive/HindiDB/data/finetuning_ready/train.json')
dev_dataset_a = load_as_hf_dataset('/content/drive/MyDrive/HindiDB/data/finetuning_ready/dev.json')

training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/HindiDB/models/qwen_ex1_createtable',
    max_steps=500,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    report_to="wandb",
    run_name="qwen-ex1-createtable-resumed",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_a,
    eval_dataset=dev_dataset_a,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=1024,
    packing=False,
)

print("✅ Trainer rebuilt fresh")

Map:   0%|          | 0/5232 [00:00<?, ? examples/s]

Map:   0%|          | 0/528 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs


✅ Trainer rebuilt fresh


In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved()/1e9:.2f} GB")
PYTORCH_CUDA_ALLOC_CONF=expandable_segments=True

GPU memory used: 3.40 GB
GPU memory reserved: 3.71 GB


In [ ]:
import torch
import numpy

# Allowlist the specific numpy function PyTorch's loader is blocking
torch.serialization.add_safe_globals([numpy._core.multiarray._reconstruct])

print("✅ Safe globals added")

✅ Safe globals added


In [ ]:
import torch

# Monkey-patch torch.load to default to weights_only=False for this session
_original_load = torch.load
def patched_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _original_load(*args, **kwargs)
torch.load = patched_load

print("✅ torch.load patched to weights_only=False")

✅ torch.load patched to weights_only=False


In [ ]:
trainer.train(resume_from_checkpoint='/content/drive/MyDrive/HindiDB/models/qwen_ex1_createtable/checkpoint-500')

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss


TrainOutput(global_step=501, training_loss=0.0002116173535525918, metrics={'train_runtime': 26.6845, 'train_samples_per_second': 299.8, 'train_steps_per_second': 18.738, 'total_flos': 7.52104707661824e+16, 'train_loss': 0.0002116173535525918, 'epoch': 1.5321100917431192})

In [ ]:
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/HindiDB/models/qwen_ex1_createtable_v2',
    max_steps=350,                    # ~1 full epoch (327 steps) + small buffer
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=14,              # 350/25 = 14, keep all checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    report_to="wandb",
    run_name="qwen-ex1-createtable-v2",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_a,
    eval_dataset=dev_dataset_a,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=1024,
    packing=False,
)

print("✅ Trainer v2 ready — full epoch coverage, eval every 25 steps")

Map:   0%|          | 0/5232 [00:00<?, ? examples/s]

Map:   0%|          | 0/528 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs


✅ Trainer v2 ready — full epoch coverage, eval every 25 steps


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
25,0.121200,1.363388
50,0.120200,1.251147
75,0.125000,1.227983
100,0.123900,1.344116
125,0.111100,1.425918
150,0.122600,1.405280
175,0.118500,1.348849
200,0.107500,1.429073


We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an excepti

In [ ]:
print(os.listdir(results_dir))

['ex1_createtable_predictions.json']


BLEU SCORING


In [ ]:
!pip install sacrebleu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.5 MB/s eta 0:00:00


In [ ]:
import json

with open('/content/drive/MyDrive/HindiDB/results/ex1_createtable_predictions.json', encoding='utf-8') as f:
    predictions = json.load(f)

print(f"Total predictions: {len(predictions)}")
print(f"\nSample:")
print(f"Prediction: {predictions[0]['prediction']}")
print(f"Gold:       {predictions[0]['gold']}")

Total predictions: 381

Sample:
Prediction: SELECT COUNT(T1.Id) FROM `Method` AS T1 INNER JOIN Solution AS T2 ON T1.SolutionId = T2.Id WHERE T1.Lang LIKE 'en' AND T1.CommentIsXml = 0;
Gold:       SELECT COUNT(Lang) FROM Method WHERE Lang = 'en' AND CommentIsXml = 1


In [ ]:
from sacrebleu.metrics import BLEU

bleu = BLEU(tokenize="none")

hypotheses = [p['prediction'] for p in predictions]
references = [[p['gold'] for p in predictions]]

bleu_score = bleu.corpus_score(hypotheses, references)
print(f"Corpus BLEU score: {bleu_score.score:.2f}")
print(bleu_score)

Corpus BLEU score: 18.20
BLEU = 18.20 34.8/20.6/14.3/10.7 (BP = 1.000 ratio = 1.668 hyp_len = 17175 ref_len = 10294)


SQAM SCORING

In [ ]:
import re

def parse_sql_clauses(sql):
    """Extract major clauses from a SQL query using regex."""
    sql = sql.strip().rstrip(';')

    clauses = {}

    # Extract each clause using regex, case-insensitive
    select_match = re.search(r'SELECT\s+(.*?)\s+FROM', sql, re.IGNORECASE | re.DOTALL)
    from_match = re.search(r'FROM\s+(.*?)(?=\s+WHERE|\s+GROUP BY|\s+HAVING|\s+ORDER BY|\s+LIMIT|$)', sql, re.IGNORECASE | re.DOTALL)
    where_match = re.search(r'WHERE\s+(.*?)(?=\s+GROUP BY|\s+HAVING|\s+ORDER BY|\s+LIMIT|$)', sql, re.IGNORECASE | re.DOTALL)
    groupby_match = re.search(r'GROUP BY\s+(.*?)(?=\s+HAVING|\s+ORDER BY|\s+LIMIT|$)', sql, re.IGNORECASE | re.DOTALL)
    having_match = re.search(r'HAVING\s+(.*?)(?=\s+ORDER BY|\s+LIMIT|$)', sql, re.IGNORECASE | re.DOTALL)
    orderby_match = re.search(r'ORDER BY\s+(.*?)(?=\s+LIMIT|$)', sql, re.IGNORECASE | re.DOTALL)

    clauses['SELECT'] = select_match.group(1).strip() if select_match else ''
    clauses['FROM'] = from_match.group(1).strip() if from_match else ''
    clauses['WHERE'] = where_match.group(1).strip() if where_match else ''
    clauses['GROUP BY'] = groupby_match.group(1).strip() if groupby_match else ''
    clauses['HAVING'] = having_match.group(1).strip() if having_match else ''
    clauses['ORDER BY'] = orderby_match.group(1).strip() if orderby_match else ''

    return clauses

def normalize_subcomponents(clause_text):
    """Split a clause into normalized subcomponents (e.g., columns), order-independent."""
    if not clause_text:
        return set()
    # Split on commas, strip whitespace, lowercase, remove table aliases for fairer comparison
    parts = [p.strip().lower() for p in clause_text.split(',')]
    # Remove alias prefixes like "T1." for comparison robustness
    parts = [re.sub(r'^t\d+\.', '', p) for p in parts]
    return set(parts)

def compute_sqam(pred_sql, gold_sql):
    """
    Computes SQAM score: weighted subcomponent match rate across SQL clauses.
    Weights follow the paper's stated hierarchy — SELECT/FROM/WHERE weighted higher
    than GROUP BY/HAVING/ORDER BY.
    """
    weights = {
        'SELECT': 0.3,
        'FROM': 0.2,
        'WHERE': 0.25,
        'GROUP BY': 0.1,
        'HAVING': 0.05,
        'ORDER BY': 0.1
    }

    pred_clauses = parse_sql_clauses(pred_sql)
    gold_clauses = parse_sql_clauses(gold_sql)

    total_score = 0.0

    for clause, weight in weights.items():
        pred_set = normalize_subcomponents(pred_clauses[clause])
        gold_set = normalize_subcomponents(gold_clauses[clause])

        if not gold_set and not pred_set:
            # Both empty — perfect match for this clause
            clause_score = 1.0
        elif not gold_set or not pred_set:
            # One empty, other isn't — no match
            clause_score = 0.0
        else:
            # Jaccard-style overlap of subcomponents
            intersection = len(pred_set & gold_set)
            union = len(pred_set | gold_set)
            clause_score = intersection / union if union > 0 else 0.0

        total_score += weight * clause_score

    return total_score

In [ ]:
import re

def split_where_clause(clause_text):
    """Splits a WHERE/HAVING clause into individual conditions on AND/OR."""
    if not clause_text:
        return set()
    # Split on AND/OR (case-insensitive), keeping conditions separate
    parts = re.split(r'\s+(?:AND|OR)\s+', clause_text, flags=re.IGNORECASE)
    parts = [p.strip().lower() for p in parts]
    # Strip table aliases (T1., T2., etc.) for fair comparison
    parts = [re.sub(r'\bt\d+\.', '', p) for p in parts]
    return set(parts)

def split_select_from_clause(clause_text):
    """Splits SELECT/FROM/GROUP BY/ORDER BY on commas, strips aliases."""
    if not clause_text:
        return set()
    parts = [p.strip().lower() for p in clause_text.split(',')]
    parts = [re.sub(r'\bt\d+\.', '', p) for p in parts]
    # Also normalize backticks and "AS aliasname" noise for FROM clause comparisons
    parts = [re.sub(r'`', '', p) for p in parts]
    return set(parts)

def compute_sqam(pred_sql, gold_sql):
    weights = {
        'SELECT': 0.3,
        'FROM': 0.2,
        'WHERE': 0.25,
        'GROUP BY': 0.1,
        'HAVING': 0.05,
        'ORDER BY': 0.1
    }

    pred_clauses = parse_sql_clauses(pred_sql)
    gold_clauses = parse_sql_clauses(gold_sql)

    total_score = 0.0
    breakdown = {}

    for clause, weight in weights.items():
        # WHERE/HAVING split on AND/OR; everything else splits on commas
        if clause in ('WHERE', 'HAVING'):
            pred_set = split_where_clause(pred_clauses[clause])
            gold_set = split_where_clause(gold_clauses[clause])
        else:
            pred_set = split_select_from_clause(pred_clauses[clause])
            gold_set = split_select_from_clause(gold_clauses[clause])

        if not gold_set and not pred_set:
            clause_score = 1.0
        elif not gold_set or not pred_set:
            clause_score = 0.0
        else:
            intersection = len(pred_set & gold_set)
            union = len(pred_set | gold_set)
            clause_score = intersection / union if union > 0 else 0.0

        breakdown[clause] = clause_score
        total_score += weight * clause_score

    return total_score, breakdown

In [ ]:
# Retest with the fix
score, breakdown = compute_sqam(test_pred, test_gold)
print(f"SQAM score: {score:.4f}")
print(f"\nBreakdown:")
for clause, s in breakdown.items():
    print(f"  {clause}: {s:.3f}")

SQAM score: 0.2500

Breakdown:
  SELECT: 0.000
  FROM: 0.000
  WHERE: 0.000
  GROUP BY: 1.000
  HAVING: 1.000
  ORDER BY: 1.000


In [ ]:
# Debug: see exactly how each clause was parsed and scored
pred_clauses = parse_sql_clauses(test_pred)
gold_clauses = parse_sql_clauses(test_gold)

print("=== PREDICTED clauses ===")
for k, v in pred_clauses.items():
    print(f"{k}: {v!r}")

print("\n=== GOLD clauses ===")
for k, v in gold_clauses.items():
    print(f"{k}: {v!r}")

print("\n=== Per-clause comparison ===")
weights = {'SELECT': 0.3, 'FROM': 0.2, 'WHERE': 0.25, 'GROUP BY': 0.1, 'HAVING': 0.05, 'ORDER BY': 0.1}
for clause, weight in weights.items():
    pred_set = normalize_subcomponents(pred_clauses[clause])
    gold_set = normalize_subcomponents(gold_clauses[clause])
    print(f"{clause} (weight {weight}): pred={pred_set}, gold={gold_set}")

=== PREDICTED clauses ===
SELECT: 'COUNT(T1.Id)'
FROM: '`Method` AS T1 INNER JOIN Solution AS T2 ON T1.SolutionId = T2.Id'
WHERE: "T1.Lang LIKE 'en' AND T1.CommentIsXml = 0"
GROUP BY: ''
HAVING: ''
ORDER BY: ''

=== GOLD clauses ===
SELECT: 'COUNT(Lang)'
FROM: 'Method'
WHERE: "Lang = 'en' AND CommentIsXml = 1"
GROUP BY: ''
HAVING: ''
ORDER BY: ''

=== Per-clause comparison ===
SELECT (weight 0.3): pred={'count(t1.id)'}, gold={'count(lang)'}
FROM (weight 0.2): pred={'`method` as t1 inner join solution as t2 on t1.solutionid = t2.id'}, gold={'method'}
WHERE (weight 0.25): pred={"lang like 'en' and t1.commentisxml = 0"}, gold={"lang = 'en' and commentisxml = 1"}
GROUP BY (weight 0.1): pred=set(), gold=set()
HAVING (weight 0.05): pred=set(), gold=set()
ORDER BY (weight 0.1): pred=set(), gold=set()


In [ ]:
pred_where = split_where_clause(pred_clauses['WHERE'])
gold_where = split_where_clause(gold_clauses['WHERE'])

print(f"Pred WHERE conditions: {pred_where}")
print(f"Gold WHERE conditions: {gold_where}")

Pred WHERE conditions: {'commentisxml = 0', "lang like 'en'"}
Gold WHERE conditions: {'commentisxml = 1', "lang = 'en'"}


In [ ]:
sqam_scores = []
breakdowns_all = []

for p in predictions:
    score, breakdown = compute_sqam(p['prediction'], p['gold'])
    sqam_scores.append(score)
    breakdowns_all.append(breakdown)

avg_sqam = sum(sqam_scores) / len(sqam_scores)
print(f"Average SQAM across {len(sqam_scores)} predictions: {avg_sqam:.4f}")

# Distribution check
import collections
print(f"\nScore distribution:")
print(f"  0.0-0.2: {sum(1 for s in sqam_scores if s < 0.2)}")
print(f"  0.2-0.4: {sum(1 for s in sqam_scores if 0.2 <= s < 0.4)}")
print(f"  0.4-0.6: {sum(1 for s in sqam_scores if 0.4 <= s < 0.6)}")
print(f"  0.6-0.8: {sum(1 for s in sqam_scores if 0.6 <= s < 0.8)}")
print(f"  0.8-1.0: {sum(1 for s in sqam_scores if s >= 0.8)}")

Average SQAM across 381 predictions: 0.2758

Score distribution:
  0.0-0.2: 168
  0.2-0.4: 105
  0.4-0.6: 67
  0.6-0.8: 35
  0.8-1.0: 6


TSED SCORING

In [ ]:
!pip install sqlglot -q

In [ ]:
import sqlglot

# Quick test — can sqlglot parse your predictions and gold queries?
try:
    parsed_pred = sqlglot.parse_one(test_pred, read="sqlite")
    parsed_gold = sqlglot.parse_one(test_gold, read="sqlite")
    print("✅ Both parsed successfully")
    print(f"\nPredicted AST:\n{parsed_pred}")
    print(f"\nGold AST:\n{repr(parsed_gold)[:300]}")
except Exception as e:
    print(f"❌ Parse error: {e}")

✅ Both parsed successfully

Predicted AST:
SELECT COUNT(T1.Id) FROM "Method" AS T1 INNER JOIN Solution AS T2 ON T1.SolutionId = T2.Id WHERE T1.Lang LIKE 'en' AND T1.CommentIsXml = 0

Gold AST:
Select(
  expressions=[
    Count(
      this=Column(
        this=Identifier(this=Lang, quoted=False)),
      big_int=True)],
  from=From(
    this=Table(
      this=Identifier(this=Method, quoted=False))),
  where=Where(
    this=And(
      this=EQ(
        this=Column(
          this=Identifier(t


In [ ]:
import sqlglot
from sqlglot import exp

def ast_to_node_list(node):
    """
    Flattens an AST into a list of (node_type, key_attributes) tuples,
    representing each node in a normalized, comparable form.
    """
    nodes = []

    def walk(n, depth=0):
        if n is None:
            return
        # Capture node type and any literal/identifier value it carries
        node_repr = type(n).__name__
        if hasattr(n, 'this') and isinstance(n.this, str):
            node_repr += f":{n.this.lower()}"
        nodes.append((depth, node_repr))

        for child in n.args.values():
            if isinstance(child, list):
                for c in child:
                    if isinstance(c, exp.Expression):
                        walk(c, depth + 1)
            elif isinstance(child, exp.Expression):
                walk(child, depth + 1)

    walk(node)
    return nodes

def tree_edit_distance(nodes_a, nodes_b):
    """
    Simplified tree edit distance via sequence-level Levenshtein distance
    over the flattened node lists (depth, type) — a practical approximation
    of full AST tree-edit-distance.
    """
    m, n = len(nodes_a), len(nodes_b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if nodes_a[i-1] == nodes_b[j-1] else 1
            dp[i][j] = min(
                dp[i-1][j] + 1,      # deletion
                dp[i][j-1] + 1,      # insertion
                dp[i-1][j-1] + cost  # substitution
            )

    return dp[m][n]

def compute_tsed(pred_sql, gold_sql):
    """
    Computes TSED = D/N where D is tree edit distance, N is node count
    of the larger tree. Returns 1 - normalized_distance (so 1.0 = perfect match).
    """
    try:
        pred_ast = sqlglot.parse_one(pred_sql, read="sqlite")
        gold_ast = sqlglot.parse_one(gold_sql, read="sqlite")
    except Exception:
        return 0.0  # unparseable prediction = worst score

    pred_nodes = ast_to_node_list(pred_ast)
    gold_nodes = ast_to_node_list(gold_ast)

    D = tree_edit_distance(pred_nodes, gold_nodes)
    N = max(len(pred_nodes), len(gold_nodes))

    if N == 0:
        return 1.0

    distance_ratio = D / N
    tsed_score = max(0.0, 1.0 - distance_ratio)  # paper defines 1=best, 0=worst

    return tsed_score

In [ ]:
import time

tsed_scores = []
start_time = time.time()

for i, p in enumerate(predictions):
    score = compute_tsed(p['prediction'], p['gold'])
    tsed_scores.append(score)

    if i % 50 == 0:
        elapsed = time.time() - start_time
        print(f"Progress: {i+1}/{len(predictions)} — {elapsed:.0f}s elapsed")

avg_tsed = sum(tsed_scores) / len(tsed_scores)
print(f"\nAverage TSED across {len(tsed_scores)} predictions: {avg_tsed:.4f}")

# Distribution check, same buckets as SQAM for comparison
print(f"\nScore distribution:")
print(f"  0.0-0.2: {sum(1 for s in tsed_scores if s < 0.2)}")
print(f"  0.2-0.4: {sum(1 for s in tsed_scores if 0.2 <= s < 0.4)}")
print(f"  0.4-0.6: {sum(1 for s in tsed_scores if 0.4 <= s < 0.6)}")
print(f"  0.6-0.8: {sum(1 for s in tsed_scores if 0.6 <= s < 0.8)}")
print(f"  0.8-1.0: {sum(1 for s in tsed_scores if s >= 0.8)}")

Progress: 1/381 — 0s elapsed
Progress: 51/381 — 0s elapsed
Progress: 101/381 — 0s elapsed
Progress: 151/381 — 1s elapsed
Progress: 201/381 — 1s elapsed
Progress: 251/381 — 1s elapsed
Progress: 301/381 — 1s elapsed
Progress: 351/381 — 2s elapsed

Average TSED across 381 predictions: 0.2894

Score distribution:
  0.0-0.2: 152
  0.2-0.4: 101
  0.4-0.6: 90
  0.6-0.8: 25
  0.8-1.0: 13


SAVING RESULTS

In [ ]:
import json

ex1_run_a_results = {
    'experiment': 'EX1 Run A',
    'model': 'Qwen-2.5-Coder-3B',
    'prompt_format': 'CREATE TABLE',
    'language': 'Hindi',
    'checkpoint': 'checkpoint-75 (best eval_loss: 1.228)',
    'test_set_size': len(predictions),
    'metrics': {
        'bleu': 18.20,
        'sqam': round(avg_sqam, 4),
        'tsed': round(avg_tsed, 4)
    },
    'comparison_to_paper_darija': {
        'bleu': {'paper': 36.2, 'yours': 18.20, 'gap_pct': -49.7},
        'sqam': {'paper': 0.483, 'yours': round(avg_sqam, 4), 'gap_pct': round((avg_sqam - 0.483)/0.483*100, 1)},
        'tsed': {'paper': 0.286, 'yours': round(avg_tsed, 4), 'gap_pct': round((avg_tsed - 0.286)/0.286*100, 1)}
    }
}

with open('/content/drive/MyDrive/HindiDB/results/ex1_run_a_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(ex1_run_a_results, f, ensure_ascii=False, indent=2)

print("✅ Saved ex1_run_a_metrics.json")
print(json.dumps(ex1_run_a_results, indent=2, ensure_ascii=False))

✅ Saved ex1_run_a_metrics.json
{
  "experiment": "EX1 Run A",
  "model": "Qwen-2.5-Coder-3B",
  "prompt_format": "CREATE TABLE",
  "language": "Hindi",
  "checkpoint": "checkpoint-75 (best eval_loss: 1.228)",
  "test_set_size": 381,
  "metrics": {
    "bleu": 18.2,
    "sqam": 0.2758,
    "tsed": 0.2894
  },
  "comparison_to_paper_darija": {
    "bleu": {
      "paper": 36.2,
      "yours": 18.2,
      "gap_pct": -49.7
    },
    "sqam": {
      "paper": 0.483,
      "yours": 0.2758,
      "gap_pct": -42.9
    },
    "tsed": {
      "paper": 0.286,
      "yours": 0.2894,
      "gap_pct": 1.2
    }
  }
}


SIMLTANEOUS EVAL OF THE CREATE TABLE AND SERIALISED FORMATS

In [ ]:
import json

# Load Run A (CREATE TABLE) predictions
with open('/content/drive/MyDrive/HindiDB/results/ex1_createtable_predictions.json', encoding='utf-8') as f:
    predictions_a = json.load(f)

# Load Run B (Serialized) predictions
with open('/content/drive/MyDrive/HindiDB/results/ex1_serialized_predictions.json', encoding='utf-8') as f:
    predictions_b = json.load(f)

print(f"Run A (CREATE TABLE): {len(predictions_a)} predictions")
print(f"Run B (Serialized):   {len(predictions_b)} predictions")

Run A (CREATE TABLE): 381 predictions
Run B (Serialized):   381 predictions


In [ ]:
!pip install sacrebleu sqlglot -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.3 MB/s eta 0:00:00


In [ ]:
# Paste all three metric functions here — BLEU, SQAM, TSED
from sacrebleu.metrics import BLEU
import re
import sqlglot
from sqlglot import exp

# ── BLEU ─────────────────────────────────────────────────────
def compute_bleu(predictions):
    bleu = BLEU(tokenize="none")
    hypotheses = [p['prediction'] for p in predictions]
    references = [[p['gold'] for p in predictions]]
    return bleu.corpus_score(hypotheses, references).score

# ── SQAM helpers ─────────────────────────────────────────────
def parse_sql_clauses(sql):
    sql = sql.strip().rstrip(';')
    clauses = {}
    patterns = {
        'SELECT':   r'SELECT\s+(.*?)\s+FROM',
        'FROM':     r'FROM\s+(.*?)(?=\s+WHERE|\s+GROUP BY|\s+HAVING|\s+ORDER BY|\s+LIMIT|$)',
        'WHERE':    r'WHERE\s+(.*?)(?=\s+GROUP BY|\s+HAVING|\s+ORDER BY|\s+LIMIT|$)',
        'GROUP BY': r'GROUP BY\s+(.*?)(?=\s+HAVING|\s+ORDER BY|\s+LIMIT|$)',
        'HAVING':   r'HAVING\s+(.*?)(?=\s+ORDER BY|\s+LIMIT|$)',
        'ORDER BY': r'ORDER BY\s+(.*?)(?=\s+LIMIT|$)',
    }
    for clause, pattern in patterns.items():
        match = re.search(pattern, sql, re.IGNORECASE | re.DOTALL)
        clauses[clause] = match.group(1).strip() if match else ''
    return clauses

def split_where_clause(text):
    if not text: return set()
    parts = re.split(r'\s+(?:AND|OR)\s+', text, flags=re.IGNORECASE)
    parts = [re.sub(r'\bt\d+\.', '', p.strip().lower()) for p in parts]
    return set(parts)

def split_select_from_clause(text):
    if not text: return set()
    parts = [re.sub(r'`', '', re.sub(r'\bt\d+\.', '', p.strip().lower())) for p in text.split(',')]
    return set(parts)

def compute_sqam(pred_sql, gold_sql):
    weights = {'SELECT':0.3,'FROM':0.2,'WHERE':0.25,'GROUP BY':0.1,'HAVING':0.05,'ORDER BY':0.1}
    pred_c = parse_sql_clauses(pred_sql)
    gold_c = parse_sql_clauses(gold_sql)
    score = 0.0
    for clause, weight in weights.items():
        if clause in ('WHERE', 'HAVING'):
            pred_set = split_where_clause(pred_c[clause])
            gold_set = split_where_clause(gold_c[clause])
        else:
            pred_set = split_select_from_clause(pred_c[clause])
            gold_set = split_select_from_clause(gold_c[clause])
        if not gold_set and not pred_set:
            clause_score = 1.0
        elif not gold_set or not pred_set:
            clause_score = 0.0
        else:
            i = len(pred_set & gold_set)
            u = len(pred_set | gold_set)
            clause_score = i / u if u > 0 else 0.0
        score += weight * clause_score
    return score

# ── TSED helpers ──────────────────────────────────────────────
def ast_to_node_list(node):
    nodes = []
    def walk(n, depth=0):
        if n is None: return
        node_repr = type(n).__name__
        if hasattr(n, 'this') and isinstance(n.this, str):
            node_repr += f":{n.this.lower()}"
        nodes.append((depth, node_repr))
        for child in n.args.values():
            if isinstance(child, list):
                for c in child:
                    if isinstance(c, exp.Expression): walk(c, depth+1)
            elif isinstance(child, exp.Expression):
                walk(child, depth+1)
    walk(node)
    return nodes

def tree_edit_distance(a, b):
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)
    return dp[m][n]

def compute_tsed(pred_sql, gold_sql):
    try:
        pred_ast = sqlglot.parse_one(pred_sql, read="sqlite")
        gold_ast = sqlglot.parse_one(gold_sql, read="sqlite")
    except Exception:
        return 0.0
    pred_nodes = ast_to_node_list(pred_ast)
    gold_nodes = ast_to_node_list(gold_ast)
    D = tree_edit_distance(pred_nodes, gold_nodes)
    N = max(len(pred_nodes), len(gold_nodes))
    return max(0.0, 1.0 - D/N) if N > 0 else 1.0

print("✅ All metric functions defined")

✅ All metric functions defined


In [ ]:
# Score both runs
def score_predictions(predictions, name):
    bleu = compute_bleu(predictions)
    sqam_scores = [compute_sqam(p['prediction'], p['gold']) for p in predictions]
    tsed_scores = [compute_tsed(p['prediction'], p['gold']) for p in predictions]
    avg_sqam = sum(sqam_scores) / len(sqam_scores)
    avg_tsed = sum(tsed_scores) / len(tsed_scores)
    print(f"\n{'='*40}")
    print(f"Results for: {name}")
    print(f"  BLEU: {bleu:.2f}")
    print(f"  SQAM: {avg_sqam:.4f}")
    print(f"  TSED: {avg_tsed:.4f}")
    return {'bleu': bleu, 'sqam': avg_sqam, 'tsed': avg_tsed}

results_a = score_predictions(predictions_a, "EX1 Run A — CREATE TABLE")
results_b = score_predictions(predictions_b, "EX1 Run B — Serialized")


Results for: EX1 Run A — CREATE TABLE
  BLEU: 18.20
  SQAM: 0.2758
  TSED: 0.2894

Results for: EX1 Run B — Serialized
  BLEU: 10.42
  SQAM: 0.2347
  TSED: 0.2158


In [ ]:
# Save complete EX1 comparison
import json, os

ex1_results = {
    'experiment': 'EX1: Prompt Design Comparison',
    'model': 'Qwen-2.5-Coder-3B',
    'language': 'Hindi',
    'runs': {
        'create_table': {
            'checkpoint': 'checkpoint-75',
            'best_eval_loss': 1.228,
            'metrics': results_a
        },
        'serialized': {
            'checkpoint': 'checkpoint-25',
            'best_eval_loss': 1.1815,
            'metrics': results_b
        }
    },
    'paper_darija_reference': {
        'create_table': {'bleu': 36.2, 'sqam': 0.483, 'tsed': 0.286},
        'serialized':   {'bleu': 32.5, 'sqam': 0.472, 'tsed': 0.220}
    },
    'winner': 'TBD — determined after scoring'
}

os.makedirs('/content/drive/MyDrive/HindiDB/results', exist_ok=True)
with open('/content/drive/MyDrive/HindiDB/results/ex1_complete.json', 'w', encoding='utf-8') as f:
    json.dump(ex1_results, f, ensure_ascii=False, indent=2)

print("✅ Saved ex1_complete.json")

✅ Saved ex1_complete.json


FRESH START ON HINGLISH

In [ ]:
!pip install google-generativeai -q

In [ ]:
!pip uninstall -y google-generativeai google-ai-generativelanguage
!pip install -U google-generativeai

Found existing installation: google-generativeai 0.8.6
Uninstalling google-generativeai-0.8.6:
  Successfully uninstalled google-generativeai-0.8.6
Found existing installation: google-ai-generativelanguage 0.6.15
Uninstalling google-ai-generativelanguage-0.6.15:
  Successfully uninstalled google-ai-generativelanguage-0.6.15
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import time
from google.colab import userdata
import google.generativeai as genai

# 1. Configure API Key
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

# 2. Initialize the Model
# Note: Swapped 'gemini-flash-latest' to 'gemini-1.5-flash' for proper API compatibility
model = genai.GenerativeModel(
    model_name="gemini-1.5-flash",
    system_instruction="""You are a Hinglish translator for database questions.
Convert English text-to-SQL questions into natural Hinglish using Roman script.

Rules:
1. Roman script ONLY — no Devanagari characters whatsoever
2. Use Hindi grammar and sentence structure with English vocabulary mixed in naturally
3. Keep all proper nouns, numbers, and database-relevant terms in English
4. Sound like a bilingual urban Indian speaker — natural, not literal
5. Match the original question length roughly
6. Return ONLY the Hinglish translation, nothing else"""
)

print("✅ Gemini model loaded")

# 3. Load your uploaded dataset (Change file name as needed)
# If it's an Excel file, use pd.read_excel('filename.xlsx') instead
df = pd.read_csv('your_dataset.csv')

# 4. Run the Translation Pipeline
hinglish_translations = []

print("🚀 Starting translation...")

for index, row in df.iterrows():
    # ⚠️ CHANGE 'english_question' to your actual column name
    english_text = row['english_question']

    try:
        # Call the Gemini API
        response = model.generate_content(english_text)
        translation = response.text.strip()
        hinglish_translations.append(translation)

        # Progress tracker printout
        print(f"[{index + 1}/{len(df)}] Success!")

    except Exception as e:
        print(f"❌ Error at index {index}: {e}")
        hinglish_translations.append("TRANSLATION_ERROR")

    # 5. Rate limiting safety pause (1 second per request)
    time.sleep(1)

# 6. Save back to a new dataset file
df['hinglish_translation'] = hinglish_translations
df.to_csv('translated_dataset.csv', index=False)

print("\n🎉 Translation complete! File saved as 'translated_dataset.csv'")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ Gemini model loaded


FileNotFoundError: [Errno 2] No such file or directory: 'your_dataset.csv'

In [ ]:
from google.colab import userdata
import google.generativeai as genai

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

model = genai.GenerativeModel(
    model_name="gemini-flash-latest",
    system_instruction="""You are a Hinglish translator for database questions.
Convert English text-to-SQL questions into natural Hinglish using Roman script.

Rules:
1. Roman script ONLY — no Devanagari characters whatsoever
2. Use Hindi grammar and sentence structure with English vocabulary mixed in naturally
3. Keep all proper nouns, numbers, and database-relevant terms in English
4. Sound like a bilingual urban Indian speaker — natural, not literal
5. Match the original question length roughly
6. Return ONLY the Hinglish translation, nothing else"""
)

print("✅ Gemini model loaded")

AttributeError: module 'google.generativeai' has no attribute 'configure'

In [ ]:
# List all currently available Gemini models
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

AttributeError: module 'google.generativeai' has no attribute 'configure'

In [ ]:
def translate_to_hinglish(question, max_retries=5):
    """Translates English to Hinglish using Gemini 1.5 Flash with retry logic."""
    for attempt in range(max_retries):
        try:
            response = model.generate_content(question)
            return response.text.strip()
        except Exception as e:
            wait_time = (2 ** attempt) + 1
            if attempt < max_retries - 1:
                print(f"  ⚠️ Attempt {attempt+1} failed: {e} — retrying in {wait_time}s")
                time.sleep(wait_time)
            else:
                print(f"  ❌ All retries failed: {question[:50]}...")
                return ''

# Quick quality test
test_questions = [
    "How many Chinese restaurants are there on 1st st, Livermore?",
    "Calculate the average age of clients from the Midwest region.",
    "List all universities that scored under 60 in teaching in 2011."
]

print("=== Quality check ===")
for q in test_questions:
    h = translate_to_hinglish(q)
    print(f"EN: {q}")
    print(f"HI: {h}\n")

=== Quality check ===


  ⚠️ Attempt 1 failed: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 10.642601477s. — retrying in 2s


  ⚠️ Attempt 2 failed: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 7.988969786s. — retrying in 3s


  ⚠️ Attempt 3 failed: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 4.307321939s. — retrying in 5s


  ⚠️ Attempt 4 failed: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
Please retry in 58.608399845s. — retrying in 9s


...

  ⚠️ Attempt 1 failed: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 26.190971946s. — retrying in 2s


  ⚠️ Attempt 2 failed: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
Please retry in 23.60902322s. — retrying in 3s


  ⚠️ Attempt 3 failed: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
Please retry in 20.014679241s. — retrying in 5s


  ⚠️ Attempt 4 failed: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 14.400568272s. — retrying in 9s


  ❌ All retries failed: List all universities that scored under 60 in teac...
EN: List all universities that scored under 60 in teaching in 2011.
HI: 



In [ ]:
model = genai.GenerativeModel(
    model_name="gemini-2.0-flash",
    system_instruction="""Translate English database questions to Hinglish (Roman script only, no Devanagari).
Use natural Hindi grammar with English technical terms. Return translation only.
Examples:
EN: How many students scored above 90? → HI: Kitne students ne 90 se zyada score kiya?
EN: List employees ordered by salary. → HI: Employees ko salary ke order mein list karo.
EN: What is the price of car ID 15? → HI: Car ID 15 ka price kya hai?"""
)

print("✅ Gemini 2.0 Flash loaded")

✅ Gemini 2.0 Flash loaded


In [ ]:
import json
import time
import pandas as pd
from google.colab import userdata
from google import genai
from google.genai import types

# 1. Initialize Client (Using modern SDK syntax)
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

config = types.GenerateContentConfig(
    system_instruction="""You are a Hinglish translator for database questions.
Convert English text-to-SQL questions into natural Hinglish using Roman script.

Rules:
1. Roman script ONLY — no Devanagari characters whatsoever
2. Use Hindi grammar and sentence structure with English vocabulary mixed in naturally
3. Keep all proper nouns, numbers, and database-relevant terms in English
4. Sound like a bilingual urban Indian speaker — natural, not literal
5. Match the original question length roughly
6. Return ONLY the Hinglish translation, nothing else""",
    temperature=0.3
)

# 2. Translation function with smarter API throttling
def translate_to_hinglish(question, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model='gemini-1.5-flash', # Hardcoding 1.5-flash can sometimes bypass 2.0 temporary locks
                contents=question,
                config=config
            )
            return response.text.strip()

        except Exception as e:
            error_msg = str(e)

            # If we hit a rate limit, force a longer wait time (Free tier cooling period)
            if "429" in error_msg or "Quota" in error_msg:
                wait_time = 12  # Standard cooling window for Gemini free tier RPM
            else:
                wait_time = (2 ** attempt) + 2

            if attempt < max_retries - 1:
                print(f"⚠️ Rate limited. Waiting {wait_time}s before attempt {attempt+2}...")
                time.sleep(wait_time)
            else:
                print(f"❌ All retries exhausted for: {question[:30]}...")
                return 'TRANSLATION_ERROR'

# === Quality Check Loop (With mandatory delay) ===
test_questions = [
    "How many Chinese restaurants are there on 1st st, Livermore?",
    "Calculate the average age of clients from the Midwest region.",
    "List all universities that scored under 60 in teaching in 2011."
]

print("=== Quality check ===")
for q in test_questions:
    h = translate_to_hinglish(q)
    print(f"EN: {q}")
    print(f"HI: {h}\n")
    time.sleep(2) # Prevent slamming the API on successful hits

=== Quality check ===
⚠️ Rate limited. Waiting 3s before attempt 2...
⚠️ Rate limited. Waiting 4s before attempt 3...
⚠️ Rate limited. Waiting 6s before attempt 4...
⚠️ Rate limited. Waiting 10s before attempt 5...
❌ All retries exhausted for: How many Chinese restaurants a...
EN: How many Chinese restaurants are there on 1st st, Livermore?
HI: TRANSLATION_ERROR

⚠️ Rate limited. Waiting 3s before attempt 2...
⚠️ Rate limited. Waiting 4s before attempt 3...
⚠️ Rate limited. Waiting 6s before attempt 4...
⚠️ Rate limited. Waiting 10s before attempt 5...
❌ All retries exhausted for: Calculate the average age of c...
EN: Calculate the average age of clients from the Midwest region.
HI: TRANSLATION_ERROR

⚠️ Rate limited. Waiting 3s before attempt 2...
⚠️ Rate limited. Waiting 4s before attempt 3...
⚠️ Rate limited. Waiting 6s before attempt 4...
⚠️ Rate limited. Waiting 10s before attempt 5...
❌ All retries exhausted for: List all universities that sco...
EN: List all universities that sc

In [ ]:
import json
import time
import pandas as pd
from google.colab import userdata
from google import genai
from google.genai import types
from google.genai.errors import APIError

# 1. Initialize the modern client
# (Ensure your Colab secret is exactly named 'GEMINI_API_KEY')
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

# 2. Set up system instructions using the new layout
config = types.GenerateContentConfig(
    system_instruction="""You are a Hinglish translator for database questions.
Convert English text-to-SQL questions into natural Hinglish using Roman script.

Rules:
1. Roman script ONLY — no Devanagari characters whatsoever
2. Use Hindi grammar and sentence structure with English vocabulary mixed in naturally
3. Keep all proper nouns, numbers, and database-relevant terms in English
4. Sound like a bilingual urban Indian speaker — natural, not literal
5. Match the original question length roughly
6. Return ONLY the Hinglish translation, nothing else""",
    temperature=0.3
)

# 3. Secure translation function with quota-safeguards
def translate_to_hinglish(question, max_retries=3):
    for attempt in range(max_retries):
        try:
            # The modern way to call the model
            response = client.models.generate_content(
                model='gemini-1.5-flash',
                contents=question,
                config=config
            )
            return response.text.strip()

        except APIError as e:
            error_msg = str(e)

            # Check if your free tier is fully exhausted (limit: 0)
            if "429" in error_msg or "Quota exceeded" in error_msg:
                print(f"\n🛑 API Quota Blocked: Your free tier limit is currently 0.")
                print("Stopping script to prevent endless loops. You may need to wait for a reset or check AI Studio.")
                return "QUOTA_EXHAUSTED"

            # Backoff for temporary hiccups
            wait_time = (2 ** attempt) + 2
            if attempt < max_retries - 1:
                print(f"⚠️ Hit a snag. Retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                return "TRANSLATION_ERROR"
        except Exception as e:
            print(f"❌ Unexpected Error: {e}")
            return "ERROR"

# === 4. Run the Quality Test ===
print("=== Quality Check ===")
test_questions = [
    "How many Chinese restaurants are there on 1st st, Livermore?",
    "Calculate the average age of clients from the Midwest region."
]

for q in test_questions:
    h = translate_to_hinglish(q)
    print(f"EN: {q}")
    print(f"HI: {h}\n")
    time.sleep(2) # Safe spacing

# === 5. Process your uploaded JSON File ===
print("\n--- Starting Dataset Processing ---")
file_path = "/content/drive/MyDrive/HindiDB/data/filtered_train.json"

try:
    df = pd.read_json(file_path)
    print(f"📂 Successfully loaded {len(df)} rows from dataset.")

    hinglish_translations = []

    for index, row in df.iterrows():
        # Using 'question' as the column based on standard text-to-SQL schemas
        english_text = row['question']

        print(f"Processing [{index + 1}/{len(df)}] -> ", end="")
        translation = translate_to_hinglish(english_text)
        print(f"Done.")

        if translation == "QUOTA_EXHAUSTED":
            print("❌ Exiting loop early due to exhausted API limits.")
            break

        hinglish_translations.append(translation)
        time.sleep(2.5) # Mandatory breathing room for free keys

    # Pad remaining rows if quota died early
    if len(hinglish_translations) < len(df):
        hinglish_translations += ["NOT_TRANSLATED"] * (len(df) - len(hinglish_translations))

    # Save the output back into your structured folder
    df['hinglish_question'] = hinglish_translations
    output_path = "HindiDB/data/hinglish_translated/filtered_train_translated.json"
    df.to_json(output_path, orient='records', indent=4)
    print(f"💾 Progress saved to: {output_path}")

except FileNotFoundError:
    print(f"❌ Could not locate the file at {file_path}. Double check your folder structure path!")

=== Quality Check ===
⚠️ Hit a snag. Retrying in 3s...
⚠️ Hit a snag. Retrying in 4s...
EN: How many Chinese restaurants are there on 1st st, Livermore?
HI: TRANSLATION_ERROR

⚠️ Hit a snag. Retrying in 3s...
⚠️ Hit a snag. Retrying in 4s...
EN: Calculate the average age of clients from the Midwest region.
HI: TRANSLATION_ERROR


--- Starting Dataset Processing ---
📂 Successfully loaded 6141 rows from dataset.
Processing [1/6141] -> ⚠️ Hit a snag. Retrying in 3s...
⚠️ Hit a snag. Retrying in 4s...
Done.
Processing [2/6141] -> ⚠️ Hit a snag. Retrying in 3s...
⚠️ Hit a snag. Retrying in 4s...
Done.
Processing [3/6141] -> ⚠️ Hit a snag. Retrying in 3s...
⚠️ Hit a snag. Retrying in 4s...
Done.
Processing [4/6141] -> ⚠️ Hit a snag. Retrying in 3s...
⚠️ Hit a snag. Retrying in 4s...
Done.
Processing [5/6141] -> ⚠️ Hit a snag. Retrying in 3s...
⚠️ Hit a snag. Retrying in 4s...
Done.
Processing [6/6141] -> ⚠️ Hit a snag. Retrying in 3s...
⚠️ Hit a snag. Retrying in 4s...
Done.
Processing [7/61

KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 7.7 MB/s eta 0:00:00


In [ ]:
!pip install groq
import os
import json
import time
import pandas as pd
from google.colab import userdata
from groq import Groq

# 1. Initialize our isolated Groq client
groq_client = Groq(api_key=userdata.get('GROQ_API_KEY'))

# 2. Strict Translation Function
def translate_to_hinglish_groq(question, max_retries=3):
    for attempt in range(max_retries):
        try:
            chat_completion = groq_client.chat.completions.create(
                messages=[
                    {
                        "role": "system",
                        "content": """You are a specialized Hinglish translation engine for text-to-SQL tasks.
CRITICAL RULES:
1. Use ROMAN SCRIPT ONLY (English alphabet). NEVER use Devanagari/Hindi characters under any circumstances.
2. Translate English text into natural, spoken Hinglish (e.g., 'Find maximum salary' -> 'Maximum salary find karo').
3. Keep all SQL keywords, column names, table names, proper nouns, and numbers strictly in English.
4. Output ONLY the raw translation string. Do not include quotes or extra conversational text.
5. Keep the structure of the sentences as a native hindi speaker."""
                    },
                    {
                        "role": "user",
                        "content": f"Translate this text: {question}"
                    }
                ],
                model="llama-3.1-8b-instant",
                temperature=0.1,  # Lower temperature prevents it from hallucinating or ignoring instructions
                max_tokens=100
            )
            return chat_completion.choices[0].message.content.strip()

        except Exception as e:
            error_msg = str(e).upper()

            # Catch Rate limits / Token exhaustion
            if "429" in error_msg or "RATE_LIMIT" in error_msg or "TPM" in error_msg:
                print("\n⏳ Tokens/Minute cap hit. Cooling down for 45 seconds to refresh your free bucket...")
                time.sleep(45)
                continue # Retries the exact same row

            time.sleep((2 ** attempt) + 2)

    return "TRANSLATION_ERROR"

# --- 3. CONFIGURE YOUR DRIVE PATHS ---
# ⚠️ REPLACE 'Your_Folder_Name' with the exact folder path where your files are in Google Drive
input_path = "/content/drive/MyDrive/HindiDB/data/filtered_train.json"
output_path = "/content/drive/MyDrive/HindiDB/data/filtered_train_translated.json"

# --- 4. SMART CHECKPOINT LOADER ---
if os.path.exists(output_path):
    print("🔄 Resuming from existing Google Drive progress checkpoint...")
    df = pd.read_json(output_path)
else:
    print("📂 Starting fresh from source dataset file...")
    df = pd.read_json(input_path)
    df['hinglish_question'] = None

# Filter for rows that haven't been successfully processed yet
to_process = df[df['hinglish_question'].isna() | (df['hinglish_question'] == 'TRANSLATION_ERROR') | (df['hinglish_question'] == '')]
print(f"📊 Dataset Total: {len(df)} rows | Remaining to translate: {len(to_process)}")

# --- 5. THE PRODUCTION PROCESSING LOOP ---
if len(to_process) > 0:
    processed_count = 0
    print("\n🎬 Pipeline starting. Watch the 'Live Proof' below to verify its correctness...")
    print("-" * 70)

    for index, row in to_process.iterrows():
        # Grabs the source English text
        english_text = row['question']

        print(f"Processing row [{index + 1}/{len(df)}] -> ", end="")

        # Core execution call
        translation = translate_to_hinglish_groq(english_text)

        # Write directly to our working DataFrame
        df.at[index, 'hinglish_question'] = translation

        # 🔬 LIVE VERIFICATION: Shows exactly what the model generated in real-time
        print(f"🔮 Live Proof: '{translation}'")

        processed_count += 1

        # Safe checkpoint auto-save directly to Drive every 15 rows
        if processed_count % 15 == 0:
            df.to_json(output_path, orient='records', indent=4)
            print(f"💾 [Drive Auto-Saved] Progress securely cached up to index {index}.")
            print("-" * 70)

        # Pacing throttle to honor Groq's 30 Requests-Per-Minute threshold safely
        time.sleep(2.2)

    # Final definitive push to drive upon completion
    df.to_json(output_path, orient='records', indent=4)
    print(f"\n🎉 SUCCESS! All rows processed. Final dataset finalized at: {output_path}")
else:
    print("\n🥳 No rows remaining to process! Everything is fully translated.")

🔄 Resuming from existing Google Drive progress checkpoint...
📊 Dataset Total: 6141 rows | Remaining to translate: 0

🥳 No rows remaining to process! Everything is fully translated.


In [ ]:
print("=== Groq Live Status Check ===")
test_queries = [
    "How many employees work in the marketing department?",
    "Find the maximum salary from the instructor table.",
    "List all orders placed in the year 2024."
]

for q in test_queries:
    result = translate_to_hinglish_groq(q)
    print(f"Original EN : {q}")
    print(f"Live Groq HI: {result}")
    print("-" * 40)
    time.sleep(2) # Normal spacing

=== Groq Live Status Check ===
Original EN : How many employees work in the marketing department?
Live Groq HI: TRANSLATION_ERROR
----------------------------------------
Original EN : Find the maximum salary from the instructor table.
Live Groq HI: TRANSLATION_ERROR
----------------------------------------
Original EN : List all orders placed in the year 2024.
Live Groq HI: TRANSLATION_ERROR
----------------------------------------


In [ ]:
# 1. Install the missing package
!pip install groq

import time
from google.colab import userdata
from groq import Groq

print("=== 🔍 Unmasking Groq API Errors ===")

test_q = "How many employees work in the marketing department?"

try:
    # 2. Use the isolated variable name to avoid Gemini conflicts
    groq_client = Groq(api_key=userdata.get('GROQ_API_KEY'))

    chat_completion = groq_client.chat.completions.create(
        messages=[
            {"role": "system", "content": "You are a Hinglish translator. Return ONLY the translation."},
            {"role": "user", "content": f"Translate this: {test_q}"}
        ],
        model="llama-3.1-8b-instant",
        temperature=0.2,
        max_tokens=100
    )
    print(f"✅ Success! Translation: {chat_completion.choices[0].message.content.strip()}")

except Exception as e:
    print("\n❌ ACTUAL API RESPONSE RECEIVED ❌")
    print("-" * 50)
    print(f"RAW ERROR MESSAGE:\n{e}")
    print("-" * 50)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.6 MB/s eta 0:00:00
=== 🔍 Unmasking Groq API Errors ===
✅ Success! Translation: कितने कर्मचारी मार्केटिंग विभाग में काम करते हैं?


TRANSLATION QUALITY CHECK

In [ ]:
import json

with open('/content/drive/MyDrive/HindiDB/data/filtered_train_translated.json', encoding='utf-8') as f:
    data = json.load(f)

print(f"Total pairs: {len(data)}")
print(f"Sample keys: {list(data[0].keys())}")
print(f"\nSample entry:")
print(f"  EN: {data[0].get('question', data[0].get('question_en', 'KEY NOT FOUND'))}")
print(f"  HI: {data[0].get('question_hinglish', 'KEY NOT FOUND')}")

Total pairs: 6141
Sample keys: ['db_id', 'question', 'evidence', 'SQL', 'hinglish_question']

Sample entry:
  EN: Name movie titles released in year 1945. Sort the listing by the descending order of movie popularity.
  HI: KEY NOT FOUND


In [ ]:
# Full quality audit
import statistics

hinglish_key = None
english_key = None

# Auto-detect key names
for key in data[0].keys():
    if 'hinglish' in key.lower():
        hinglish_key = key
    if key == 'question' or key == 'question_en':
        english_key = key

print(f"Detected keys — English: '{english_key}', Hinglish: '{hinglish_key}'")

# Collect all translations
english_lens = []
hinglish_lens = []
issues = {
    'empty': [],
    'too_short': [],        # under 5 chars
    'too_long': [],         # over 3x the English length
    'devanagari': [],       # contains Devanagari script
    'unchanged': [],        # identical to English input
}

for i, row in enumerate(data):
    en = row.get(english_key, '')
    hi = row.get(hinglish_key, '')

    english_lens.append(len(en))
    hinglish_lens.append(len(hi))

    if not hi or hi.strip() == '':
        issues['empty'].append(i)
    elif len(hi.strip()) < 5:
        issues['too_short'].append((i, hi))
    elif len(hi) > len(en) * 3:
        issues['too_long'].append((i, en[:60], hi[:60]))
    elif any('\u0900' <= c <= '\u097F' for c in hi):
        issues['devanagari'].append((i, hi[:80]))
    elif hi.strip().lower() == en.strip().lower():
        issues['unchanged'].append((i, en[:60]))

print(f"\n=== Length statistics ===")
print(f"English  — avg: {statistics.mean(english_lens):.0f} chars, "
      f"min: {min(english_lens)}, max: {max(english_lens)}")
print(f"Hinglish — avg: {statistics.mean(hinglish_lens):.0f} chars, "
      f"min: {min(hinglish_lens)}, max: {max(hinglish_lens)}")

print(f"\n=== Issues found ===")
for issue_type, items in issues.items():
    print(f"  {issue_type}: {len(items)}")

Detected keys — English: 'question', Hinglish: 'hinglish_question'

=== Length statistics ===
English  — avg: 82 chars, min: 27, max: 325
Hinglish — avg: 80 chars, min: 24, max: 348

=== Issues found ===
  empty: 0
  too_short: 0
  too_long: 1
  devanagari: 0
  unchanged: 0


In [ ]:
# Show examples of each issue type for manual inspection
for issue_type, items in issues.items():
    if items:
        print(f"\n--- {issue_type.upper()} (showing first 5) ---")
        for item in items[:5]:
            if issue_type == 'empty':
                print(f"  Index {item}: EN = {data[item].get(english_key, '')[:80]}")
            elif issue_type == 'devanagari':
                print(f"  Index {item[0]}: {item[1]}")
            elif issue_type == 'unchanged':
                print(f"  Index {item[0]}: {item[1]}")
            else:
                print(f"  Index {item[0]}: EN={item[1]} | HI={item[2]}")


--- TOO_LONG (showing first 5) ---
  Index 482: EN=What are the ids of the train running east? | HI=Train east direction mein chal raha hai, uska id kya hai? 




In [ ]:
# Fix key detection and do final clean verification
hinglish_key = 'hinglish_question'
english_key = 'question'

# Random sample check — 10 examples across different databases
import random
random.seed(99)
sample = random.sample(data, 10)

print("=== Random sample quality check ===\n")
for row in sample:
    print(f"DB:  {row['db_id']}")
    print(f"EN:  {row[english_key]}")
    print(f"HI:  {row[hinglish_key]}")
    print()

=== Random sample quality check ===

DB:  address
EN:  How many post offices are there in New York?
HI:  New York mein kitne post office hain?

DB:  legislator
EN:  What is the google entity ID of current legislator Sherrod Brown?
HI:  Google entity ID current legislator Sherrod Brown ka kya hai?

DB:  app_store
EN:  How many apps have rating of 5?
HI:  Kitne apps 5 rating ke hain?

DB:  retails
EN:  How many customers are in the furniture segment?
HI:  Kitaabon ki segment mein kitne customers hain?

DB:  movielens
EN:  What is the total amount male and female actors who were casted in movie ID 1684910 and what is the proportion between the highest quality actors against the worst quality of actors? Indicate your answer in percentage. List the the director as well as the genre.
HI:  Movie ID 1684910 mein cast hue male aur female actors ke total amount kya hai aur sabse achchi quality wale actors ke beech sabse buri quality wale actors ke beech proportion kya hai? Iska percentage mein b

In [ ]:
# Check database coverage — confirm all 55 DBs are represented
dbs_in_data = set(row['db_id'] for row in data)
print(f"Databases covered: {len(dbs_in_data)}")
print(f"DB list: {sorted(dbs_in_data)}")

Databases covered: 55
DB list: ['address', 'airline', 'app_store', 'authors', 'beer_factory', 'bike_share_1', 'car_retails', 'cars', 'chicago_crime', 'citeseer', 'codebase_comments', 'coinmarketcap', 'college_completion', 'computer_student', 'cookbook', 'craftbeer', 'cs_semester', 'disney', 'donor', 'european_football_1', 'food_inspection', 'food_inspection_2', 'genes', 'human_resources', 'ice_hockey_draft', 'image_and_language', 'language_corpus', 'law_episode', 'legislator', 'mental_health_survey', 'menu', 'movie', 'movie_platform', 'movielens', 'music_platform_2', 'music_tracker', 'regional_sales', 'restaurant', 'retail_complains', 'retail_world', 'retails', 'sales', 'sales_in_weather', 'shakespeare', 'shipping', 'shooting', 'simpson_episodes', 'social_media', 'software_company', 'superstore', 'trains', 'university', 'video_games', 'world', 'world_development_indicators']


In [ ]:
# Save a clean summary of what we have
import json

summary = {
    'dataset': 'HinglishDB',
    'total_pairs': len(data),
    'databases': len(dbs_in_data),
    'translation_model': 'llama-3.1-8b-instant (Groq)',
    'source': 'BIRD benchmark',
    'script': 'Roman (Hinglish)',
    'quality_check': {
        'empty_translations': 0,
        'devanagari_leaks': 0,
        'unchanged_from_english': 0,
        'flagged_too_long': 1,
        'verdict': 'PASS'
    }
}

with open('/content/drive/MyDrive/HindiDB/data/hinglishdb_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("✅ Summary saved")
print(json.dumps(summary, indent=2))

✅ Summary saved
{
  "dataset": "HinglishDB",
  "total_pairs": 6141,
  "databases": 55,
  "translation_model": "llama-3.1-8b-instant (Groq)",
  "source": "BIRD benchmark",
  "script": "Roman (Hinglish)",
  "quality_check": {
    "empty_translations": 0,
    "devanagari_leaks": 0,
    "unchanged_from_english": 0,
    "flagged_too_long": 1,
    "verdict": "PASS"
  }
}


In [ ]:
import os
import json

# Check what dev and test files exist and their current state
data_dir = '/content/drive/MyDrive/HindiDB/data'

# List all JSON files in the data directory
print("Files in data directory:")
for f in os.listdir(data_dir):
    if f.endswith('.json'):
        path = f'{data_dir}/{f}'
        with open(path, encoding='utf-8') as fh:
            d = json.load(fh)
        if isinstance(d, list):
            empty = sum(1 for r in d if not r.get('hinglish_question', r.get('question_hinglish', '')))
            print(f"  {f}: {len(d)} pairs, {empty} empty/missing hinglish")
        else:
            print(f"  {f}: (dict, not list)")

Files in data directory:
  filtered_train.json: 6141 pairs, 6141 empty/missing hinglish
  filtered_train_translated.json: 6141 pairs, 0 empty/missing hinglish
  hinglishdb_summary.json: (dict, not list)


In [ ]:
import json

# Load the translated data and split info
with open('/content/drive/MyDrive/HindiDB/data/filtered_train_translated.json', encoding='utf-8') as f:
    translated_data = json.load(f)

with open('/content/drive/MyDrive/HindiDB/data/splits/split_info.json') as f:
    split_info = json.load(f)

train_dbs = set(split_info['train_dbs'])
dev_dbs   = set(split_info['dev_dbs'])
test_dbs  = set(split_info['test_dbs'])

print(f"Total translated pairs: {len(translated_data)}")
print(f"Train DBs: {len(train_dbs)}")
print(f"Dev DBs:   {sorted(dev_dbs)}")
print(f"Test DBs:  {sorted(test_dbs)}")

Total translated pairs: 6141
Train DBs: 45
Dev DBs:   ['authors', 'chicago_crime', 'citeseer', 'computer_student', 'software_company']
Test DBs:  ['airline', 'codebase_comments', 'cookbook', 'mental_health_survey', 'movie']


In [ ]:
# Apply splits
train_split = [r for r in translated_data if r['db_id'] in train_dbs]
dev_split   = [r for r in translated_data if r['db_id'] in dev_dbs]
test_split  = [r for r in translated_data if r['db_id'] in test_dbs]

print(f"Train: {len(train_split)} pairs")
print(f"Dev:   {len(dev_split)} pairs")
print(f"Test:  {len(test_split)} pairs")
print(f"Total: {len(train_split)+len(dev_split)+len(test_split)} (should be 6141)")

# Verify zero overlap
train_db_set = {r['db_id'] for r in train_split}
dev_db_set   = {r['db_id'] for r in dev_split}
test_db_set  = {r['db_id'] for r in test_split}

assert train_db_set & dev_db_set == set(),  "❌ Train/Dev overlap!"
assert train_db_set & test_db_set == set(), "❌ Train/Test overlap!"
assert dev_db_set & test_db_set == set(),   "❌ Dev/Test overlap!"
print("\n✅ Zero overlap confirmed")

Train: 5232 pairs
Dev:   528 pairs
Test:  381 pairs
Total: 6141 (should be 6141)

✅ Zero overlap confirmed


In [ ]:
# Save all three splits
splits_out_dir = '/content/drive/MyDrive/HindiDB/data/hinglish_splits'
os.makedirs(splits_out_dir, exist_ok=True)

for split_name, split_data in [('train', train_split), ('dev', dev_split), ('test', test_split)]:
    out_path = f'{splits_out_dir}/{split_name}_hinglish.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(split_data, f, ensure_ascii=False, indent=2)
    print(f"✅ Saved {split_name}_hinglish.json — {len(split_data)} pairs")

✅ Saved train_hinglish.json — 5232 pairs
✅ Saved dev_hinglish.json — 528 pairs
✅ Saved test_hinglish.json — 381 pairs


In [ ]:
# Quick sanity check on each split
for split_name in ['train', 'dev', 'test']:
    path = f'{splits_out_dir}/{split_name}_hinglish.json'
    with open(path, encoding='utf-8') as f:
        d = json.load(f)
    empty = sum(1 for r in d if not r.get('hinglish_question', ''))
    dbs = {r['db_id'] for r in d}
    print(f"{split_name}: {len(d)} pairs, {empty} empty, {len(dbs)} databases")

train: 5232 pairs, 0 empty, 45 databases
dev: 528 pairs, 0 empty, 5 databases
test: 381 pairs, 0 empty, 5 databases


In [ ]:
import json

# Check dev and test splits for hinglish_question presence
for split_name in ['dev', 'test']:
    with open(f'/content/drive/MyDrive/HindiDB/data/hinglish_splits/{split_name}_hinglish.json', encoding='utf-8') as f:
        data = json.load(f)

    empty = sum(1 for r in data if not r.get('hinglish_question', '').strip())
    has_key = sum(1 for r in data if 'hinglish_question' in r)

    print(f"{split_name}: {len(data)} pairs")
    print(f"  Has 'hinglish_question' key: {has_key}")
    print(f"  Empty/missing translations: {empty}")
    print(f"  Sample: {data[10].get('hinglish_question', 'KEY MISSING')[:80]}")
    print()

dev: 528 pairs
  Has 'hinglish_question' key: 528
  Empty/missing translations: 0
  Sample: Student ID aur program years list karo jahan advisor ID 5 ke students hain.

test: 381 pairs
  Has 'hinglish_question' key: 381
  Empty/missing translations: 0
  Sample: Solution path kya hai jo maravillas_linq-to-delicious\tasty.sln hai, uska tokeni



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sqlite3, os, json

db_base = '/content/drive/MyDrive/HindiDB/data/bird_raw/train/train_databases'
splits_dir = '/content/drive/MyDrive/HindiDB/data/hinglish_splits'
ft_dir = '/content/drive/MyDrive/HindiDB/data/hinglish_finetuning_ready'
os.makedirs(ft_dir, exist_ok=True)

def get_schema_ddl(db_id):
    db_path = os.path.join(db_base, db_id, f'{db_id}.sqlite')
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT sql FROM sqlite_master WHERE type='table' AND sql IS NOT NULL")
    statements = [row[0] for row in cursor.fetchall()]
    conn.close()
    return ";\n".join(statements) + ";"

def build_hinglish_prompt(db_id, hinglish_question):
    schema = get_schema_ddl(db_id)
    return f"""{schema}

-- Using valid SQLite, answer the following Hinglish question for the tables provided above.

-- {hinglish_question}

"""

def assemble_dataset(split_name):
    with open(f'{splits_dir}/{split_name}_hinglish.json', encoding='utf-8') as f:
        data = json.load(f)

    results, skipped = [], []
    for row in data:
        try:
            prompt = build_hinglish_prompt(row['db_id'], row['hinglish_question'])
            results.append({
                'db_id': row['db_id'],
                'prompt': prompt,
                'completion': row['SQL'],
                'question_hinglish': row['hinglish_question'],
                'question_en': row['question']
            })
        except Exception as e:
            skipped.append((row['db_id'], str(e)))

    print(f"{split_name}: {len(results)}/{len(data)} built, {len(skipped)} skipped")
    return results

# Build all three splits
for split_name in ['train', 'dev', 'test']:
    data = assemble_dataset(split_name)
    out_path = f'{ft_dir}/{split_name}.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"  ✅ Saved {out_path}")

train: 5232/5232 built, 0 skipped
  ✅ Saved /content/drive/MyDrive/HindiDB/data/hinglish_finetuning_ready/train.json
dev: 528/528 built, 0 skipped
  ✅ Saved /content/drive/MyDrive/HindiDB/data/hinglish_finetuning_ready/dev.json
test: 381/381 built, 0 skipped
  ✅ Saved /content/drive/MyDrive/HindiDB/data/hinglish_finetuning_ready/test.json


In [ ]:
# Verify and preview
print("\n=== Verification ===")
for split_name in ['train', 'dev', 'test']:
    with open(f'{ft_dir}/{split_name}.json', encoding='utf-8') as f:
        d = json.load(f)
    print(f"{split_name}: {len(d)} pairs")

# Preview one sample
with open(f'{ft_dir}/train.json', encoding='utf-8') as f:
    sample = json.load(f)[0]

print(f"\n=== Sample prompt (last 200 chars) ===")
print(sample['prompt'][-200:])
print(f"\n=== Hinglish question ===")
print(sample['question_hinglish'])
print(f"\n=== Target SQL ===")
print(sample['completion'][:100])


=== Verification ===
train: 5232 pairs
dev: 528 pairs
test: 381 pairs

=== Sample prompt (last 200 chars) ===
ser_id)
);

-- Using valid SQLite, answer the following Hinglish question for the tables provided above.

-- Movie titles 1945 mein release hue the, popularity ke descending order mein listing karo.



=== Hinglish question ===
Movie titles 1945 mein release hue the, popularity ke descending order mein listing karo.

=== Target SQL ===
SELECT movie_title FROM movies WHERE movie_release_year = 1945 ORDER BY movie_popularity DESC LIMIT 


In [ ]:
import shutil, os

# Package only what's needed for Kaggle
kaggle_pkg = '/content/kaggle_hinglish'
os.makedirs(kaggle_pkg, exist_ok=True)

shutil.copytree(
    '/content/drive/MyDrive/HindiDB/data/hinglish_finetuning_ready',
    f'{kaggle_pkg}/hinglish_finetuning_ready'
)

# Zip it
shutil.make_archive('/content/drive/MyDrive/HinglishDB_finetuning', 'zip', kaggle_pkg)
print("✅ Zipped — download from Drive and upload to Kaggle Datasets")

✅ Zipped — download from Drive and upload to Kaggle Datasets


Serialised prompt for hinglish data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sqlite3, os, json

db_base = '/content/drive/MyDrive/HindiDB/data/bird_raw/train/train_databases'
splits_dir = '/content/drive/MyDrive/HindiDB/data/hinglish_splits'
serialized_dir = '/content/drive/MyDrive/HindiDB/data/hinglish_finetuning_serialized'
os.makedirs(serialized_dir, exist_ok=True)

def get_schema_serialized(db_id):
    """Compact pipe-separated schema — no types, no FK info."""
    db_path = os.path.join(db_base, db_id, f'{db_id}.sqlite')
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    table_names = [row[0] for row in cursor.fetchall()]
    table_parts = []
    for table in table_names:
        cursor.execute(f"PRAGMA table_info('{table}')")
        columns = [row[1] for row in cursor.fetchall()]
        table_parts.append(f"{table}: {', '.join(columns)}")
    conn.close()
    return f"{db_id} | " + " | ".join(table_parts)

def build_serialized_hinglish_prompt(db_id, hinglish_question):
    schema = get_schema_serialized(db_id)
    return f"""{schema}

-- Using valid SQLite, answer the following Hinglish question for the tables provided above.

-- {hinglish_question}

"""

def assemble_serialized_hinglish(split_name):
    with open(f'{splits_dir}/{split_name}_hinglish.json', encoding='utf-8') as f:
        data = json.load(f)
    results, skipped = [], []
    for row in data:
        try:
            prompt = build_serialized_hinglish_prompt(
                row['db_id'], row['hinglish_question']
            )
            results.append({
                'db_id': row['db_id'],
                'prompt': prompt,
                'completion': row['SQL'],
                'question_hinglish': row['hinglish_question'],
                'question_en': row['question']
            })
        except Exception as e:
            skipped.append((row['db_id'], str(e)))
    print(f"{split_name}: {len(results)}/{len(data)} built, {len(skipped)} skipped")
    return results

# Build all three splits
for split_name in ['train', 'dev', 'test']:
    data = assemble_serialized_hinglish(split_name)
    with open(f'{serialized_dir}/{split_name}.json', 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"  ✅ Saved {split_name}.json")

train: 5232/5232 built, 0 skipped
  ✅ Saved train.json
dev: 528/528 built, 0 skipped
  ✅ Saved dev.json
test: 381/381 built, 0 skipped
  ✅ Saved test.json


In [ ]:
# Quick verification
for split_name in ['train', 'dev', 'test']:
    with open(f'{serialized_dir}/{split_name}.json', encoding='utf-8') as f:
        d = json.load(f)
    print(f"{split_name}: {len(d)} pairs")

# Preview one example
sample = json.load(open(f'{serialized_dir}/train.json', encoding='utf-8'))[0]
print(f"\n=== Sample serialized prompt ===")
print(sample['prompt'][:300])
print(f"\nHinglish Q: {sample['question_hinglish']}")
print(f"SQL: {sample['completion'][:80]}")

train: 5232 pairs
dev: 528 pairs
test: 381 pairs

=== Sample serialized prompt ===
movie_platform | lists: user_id, list_id, list_title, list_movie_number, list_update_timestamp_utc, list_creation_timestamp_utc, list_followers, list_url, list_comments, list_description, list_cover_image_url, list_first_image_url, list_second_image_url, list_third_image_url | movies: movie_id, movi

Hinglish Q: Movie titles 1945 mein release hue the, popularity ke descending order mein listing karo.
SQL: SELECT movie_title FROM movies WHERE movie_release_year = 1945 ORDER BY movie_po


In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
# Install fine-tuning dependencies
!pip install "transformers==4.43.4" -q
!pip install "peft==0.11.1" -q
!pip install "bitsandbytes==0.49.2" --no-cache-dir -q
!pip install "accelerate==0.30.0" -q
!pip install "trl==0.8.6" -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 191.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.4/302.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 22.2 MB/s eta 0:00:00


In [ ]:
import torch, transformers, peft, trl, accelerate, bitsandbytes
print(f"CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0)}")
print(f"transformers: {transformers.__version__}")
print(f"peft:         {peft.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")

CUDA: True, GPU: Tesla T4
transformers: 4.43.4
peft:         0.11.1
bitsandbytes: 0.49.2


In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))
print("✅ Logged in")

✅ Logged in


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

model_name = "Qwen/Qwen2.5-Coder-3B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/139 [00:00<?, ?B/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
GPU memory: 3.40 GB


In [ ]:
import json
from datasets import Dataset

serialized_dir = '/content/drive/MyDrive/HindiDB/data/hinglish_finetuning_serialized'

def load_as_hf_dataset(path):
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    formatted = []
    for row in data:
        full_text = row['prompt'] + row['completion'] + tokenizer.eos_token
        formatted.append({
            'text': full_text,
            'prompt': row['prompt'],
            'completion': row['completion']
        })
    return Dataset.from_list(formatted)

train_serialized = load_as_hf_dataset(f'{serialized_dir}/train.json')
dev_serialized   = load_as_hf_dataset(f'{serialized_dir}/dev.json')

print(f"Train: {len(train_serialized)} | Dev: {len(dev_serialized)}")

Train: 5232 | Dev: 528


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
import os

checkpoint_dir = '/content/drive/MyDrive/HindiDB/models/qwen_hinglish_serialized'
os.makedirs(checkpoint_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=checkpoint_dir,
    max_steps=400,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=16,
    load_best_model_at_end=False,  # manual selection
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_serialized,
    eval_dataset=dev_serialized,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=1024,
    packing=False,
    # no EarlyStoppingCallback — you stop manually
)

print("✅ Trainer ready — no early stopping, manual checkpoint selection")
print("Watch the eval_loss column — stop the cell manually when it starts rising")
print(f"Checkpoints saving to Drive: {checkpoint_dir}")

Map:   0%|          | 0/5232 [00:00<?, ? examples/s]

Map:   0%|          | 0/528 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs


✅ Trainer ready — no early stopping, manual checkpoint selection
Watch the eval_loss column — stop the cell manually when it starts rising
Checkpoints saving to Drive: /content/drive/MyDrive/HindiDB/models/qwen_hinglish_serialized


In [ ]:
import shutil, os

# Package serialized hinglish finetuning data
kaggle_pkg = '/content/kaggle_hinglish_serialized'
if os.path.exists(kaggle_pkg):
    shutil.rmtree(kaggle_pkg)
os.makedirs(kaggle_pkg, exist_ok=True)

shutil.copytree(
    '/content/drive/MyDrive/HindiDB/data/hinglish_finetuning_serialized',
    f'{kaggle_pkg}/hinglish_finetuning_serialized'
)

shutil.make_archive(
    '/content/drive/MyDrive/HinglishDB_serialized',
    'zip',
    kaggle_pkg
)

size = os.path.getsize('/content/drive/MyDrive/HinglishDB_serialized.zip') / 1e6
print(f"✅ Zipped — {size:.1f} MB")
print("Download HinglishDB_serialized.zip from Drive and upload to Kaggle as a new dataset")

✅ Zipped — 0.6 MB
Download HinglishDB_serialized.zip from Drive and upload to Kaggle as a new dataset


In [ ]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
25,1.428600,1.402156
50,0.498500,1.685856
75,0.408000,1.747564
100,0.363000,2.029748
125,0.317000,2.133224
150,0.341800,2.093927
175,0.330600,2.251162


We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an excepti

KeyboardInterrupt: 

In [ ]:
# After training completes — print full eval_loss history for manual checkpoint selection
import json

print("=== Full eval_loss history ===\n")
checkpoints = sorted([
    c for c in os.listdir(checkpoint_dir)
    if c.startswith('checkpoint')
])

for ckpt in checkpoints:
    state_path = f'{checkpoint_dir}/{ckpt}/trainer_state.json'
    if os.path.exists(state_path):
        with open(state_path) as f:
            state = json.load(f)
        for entry in state['log_history']:
            if 'eval_loss' in entry:
                print(f"  Step {entry['step']:4d}: eval_loss = {entry['eval_loss']:.4f}")
        break  # trainer_state.json is the same across all checkpoints

print(f"\nAvailable checkpoints: {checkpoints}")
print("\nChoose your checkpoint based on the lowest eval_loss above.")

=== Full eval_loss history ===

  Step   25: eval_loss = 1.4022
  Step   50: eval_loss = 1.6859
  Step   75: eval_loss = 1.7476
  Step  100: eval_loss = 2.0297

Available checkpoints: ['checkpoint-100', 'checkpoint-125', 'checkpoint-150', 'checkpoint-175', 'checkpoint-25', 'checkpoint-50', 'checkpoint-75']

Choose your checkpoint based on the lowest eval_loss above.


In [ ]:
# Confirm checkpoint-25 exists
import os

checkpoint_dir = '/content/drive/MyDrive/HindiDB/models/qwen_hinglish_serialized'
checkpoints = sorted([c for c in os.listdir(checkpoint_dir) if c.startswith('checkpoint')])
print("Available checkpoints:", checkpoints)

Available checkpoints: ['checkpoint-100', 'checkpoint-125', 'checkpoint-150', 'checkpoint-175', 'checkpoint-25', 'checkpoint-50', 'checkpoint-75']


In [ ]:
# Load checkpoint-25 for generation
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch, gc

try:
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()
except: pass

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-3B", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-Coder-3B",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = PeftModel.from_pretrained(
    base_model,
    f'{checkpoint_dir}/checkpoint-25'
)
model.eval()
print(f"✅ Loaded checkpoint-25 (eval_loss: 1.4022)")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✅ Loaded checkpoint-25 (eval_loss: 1.4022)
GPU memory: 6.35 GB


In [ ]:
# Load serialized test set and generate predictions
import json, time

with open('/content/drive/MyDrive/HindiDB/data/hinglish_finetuning_serialized/test.json', encoding='utf-8') as f:
    test_serialized = json.load(f)

def generate_sql(prompt, max_new_tokens=150):
    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=1024
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    generated_sql = tokenizer.decode(new_tokens, skip_special_tokens=True)
    for special in ['<|fim_middle|>','<|fim_prefix|>','<|fim_suffix|>',
                    '<|endoftext|>','<|im_end|>']:
        generated_sql = generated_sql.replace(special, '')
    return generated_sql.split('\n')[0].strip()

predictions_serialized = []
start = time.time()

for i, row in enumerate(test_serialized):
    pred = generate_sql(row['prompt'])
    predictions_serialized.append({
        'db_id': row['db_id'],
        'question_hinglish': row['question_hinglish'],
        'prediction': pred,
        'gold': row['completion']
    })
    if i % 50 == 0:
        print(f"Progress: {i+1}/{len(test_serialized)} — {time.time()-start:.0f}s")

print(f"✅ Done — {len(predictions_serialized)} predictions")

# Save immediately
import os
os.makedirs('/content/drive/MyDrive/HindiDB/results', exist_ok=True)
with open('/content/drive/MyDrive/HindiDB/results/hinglish_serialized_predictions.json',
          'w', encoding='utf-8') as f:
    json.dump(predictions_serialized, f, ensure_ascii=False, indent=2)
print("✅ Saved to Drive")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Progress: 1/381 — 14s
Progress: 51/381 — 760s
Progress: 101/381 — 1478s
Progress: 151/381 — 2208s
Progress: 201/381 — 3033s
Progress: 251/381 — 4068s
Progress: 301/381 — 4951s
Progress: 351/381 — 5752s
✅ Done — 381 predictions
✅ Saved to Drive


In [ ]:
!pip install sacrebleu sqlglot -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.1 MB/s eta 0:00:00


In [ ]:
import json
import re
import sqlglot
from sqlglot import exp
from sacrebleu.metrics import BLEU

# Load predictions
with open('/content/drive/MyDrive/HindiDB/results/hinglish_serialized_predictions.json', encoding='utf-8') as f:
    predictions_serialized = json.load(f)

print(f"Loaded {len(predictions_serialized)} predictions")

# ── Metric functions ─────────────────────────────────────────
def compute_bleu(predictions):
    bleu = BLEU(tokenize="none")
    hypotheses = [p['prediction'] for p in predictions]
    references = [[p['gold'] for p in predictions]]
    return bleu.corpus_score(hypotheses, references).score

def parse_sql_clauses(sql):
    sql = sql.strip().rstrip(';')
    patterns = {
        'SELECT':   r'SELECT\s+(.*?)\s+FROM',
        'FROM':     r'FROM\s+(.*?)(?=\s+WHERE|\s+GROUP BY|\s+HAVING|\s+ORDER BY|\s+LIMIT|$)',
        'WHERE':    r'WHERE\s+(.*?)(?=\s+GROUP BY|\s+HAVING|\s+ORDER BY|\s+LIMIT|$)',
        'GROUP BY': r'GROUP BY\s+(.*?)(?=\s+HAVING|\s+ORDER BY|\s+LIMIT|$)',
        'HAVING':   r'HAVING\s+(.*?)(?=\s+ORDER BY|\s+LIMIT|$)',
        'ORDER BY': r'ORDER BY\s+(.*?)(?=\s+LIMIT|$)',
    }
    clauses = {}
    for clause, pattern in patterns.items():
        match = re.search(pattern, sql, re.IGNORECASE | re.DOTALL)
        clauses[clause] = match.group(1).strip() if match else ''
    return clauses

def split_where_clause(text):
    if not text: return set()
    parts = re.split(r'\s+(?:AND|OR)\s+', text, flags=re.IGNORECASE)
    return set(re.sub(r'\bt\d+\.', '', p.strip().lower()) for p in parts)

def split_select_from_clause(text):
    if not text: return set()
    return set(
        re.sub(r'`', '', re.sub(r'\bt\d+\.', '', p.strip().lower()))
        for p in text.split(',')
    )

def compute_sqam(pred_sql, gold_sql):
    weights = {
        'SELECT': 0.3, 'FROM': 0.2, 'WHERE': 0.25,
        'GROUP BY': 0.1, 'HAVING': 0.05, 'ORDER BY': 0.1
    }
    pred_c = parse_sql_clauses(pred_sql)
    gold_c = parse_sql_clauses(gold_sql)
    score = 0.0
    for clause, weight in weights.items():
        if clause in ('WHERE', 'HAVING'):
            pred_set = split_where_clause(pred_c[clause])
            gold_set = split_where_clause(gold_c[clause])
        else:
            pred_set = split_select_from_clause(pred_c[clause])
            gold_set = split_select_from_clause(gold_c[clause])
        if not gold_set and not pred_set:
            clause_score = 1.0
        elif not gold_set or not pred_set:
            clause_score = 0.0
        else:
            i = len(pred_set & gold_set)
            u = len(pred_set | gold_set)
            clause_score = i/u if u > 0 else 0.0
        score += weight * clause_score
    return score

def ast_to_node_list(node):
    nodes = []
    def walk(n, depth=0):
        if n is None: return
        node_repr = type(n).__name__
        if hasattr(n, 'this') and isinstance(n.this, str):
            node_repr += f":{n.this.lower()}"
        nodes.append((depth, node_repr))
        for child in n.args.values():
            if isinstance(child, list):
                for c in child:
                    if isinstance(c, exp.Expression): walk(c, depth+1)
            elif isinstance(child, exp.Expression):
                walk(child, depth+1)
    walk(node)
    return nodes

def tree_edit_distance(a, b):
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            cost = 0 if a[i-1]==b[j-1] else 1
            dp[i][j] = min(
                dp[i-1][j]+1,
                dp[i][j-1]+1,
                dp[i-1][j-1]+cost
            )
    return dp[m][n]

def compute_tsed(pred_sql, gold_sql):
    try:
        pred_ast = sqlglot.parse_one(pred_sql, read="sqlite")
        gold_ast = sqlglot.parse_one(gold_sql, read="sqlite")
    except Exception:
        return 0.0
    pred_nodes = ast_to_node_list(pred_ast)
    gold_nodes = ast_to_node_list(gold_ast)
    D = tree_edit_distance(pred_nodes, gold_nodes)
    N = max(len(pred_nodes), len(gold_nodes))
    return max(0.0, 1.0 - D/N) if N > 0 else 1.0

print("✅ All metric functions defined")

Loaded 381 predictions
✅ All metric functions defined


In [ ]:
# Score serialized predictions
bleu_s = compute_bleu(predictions_serialized)
sqam_s = [compute_sqam(p['prediction'], p['gold']) for p in predictions_serialized]
tsed_s = [compute_tsed(p['prediction'], p['gold']) for p in predictions_serialized]

avg_sqam_s = sum(sqam_s)/len(sqam_s)
avg_tsed_s = sum(tsed_s)/len(tsed_s)

# CREATE TABLE scores (already have these)
bleu_ct  = 20.39
sqam_ct  = 0.319
tsed_ct  = 0.308

print("=" * 60)
print("EX1: Prompt Design — Hinglish (Qwen-2.5-Coder-3B)")
print("=" * 60)
print(f"\n{'Prompt':<20} {'BLEU':>8} {'SQAM':>8} {'TSED':>8}")
print("-" * 48)
print(f"{'CREATE TABLE':<20} {bleu_ct:>8.2f} {sqam_ct:>8.4f} {tsed_ct:>8.4f}")
print(f"{'Serialized':<20} {bleu_s:>8.2f} {avg_sqam_s:>8.4f} {avg_tsed_s:>8.4f}")
print("-" * 48)
print(f"\nDarijaDB reference (Qwen-2.5-Coder-3B):")
print(f"{'CREATE TABLE':<20} {'36.20':>8} {'0.483':>8} {'0.286':>8}")
print(f"{'Serialized':<20} {'32.50':>8} {'0.472':>8} {'0.220':>8}")

# Save results
ex1_results = {
    'experiment': 'EX1 — Prompt Design Comparison',
    'model': 'Qwen-2.5-Coder-3B',
    'dataset': 'HinglishDB',
    'test_set_size': len(predictions_serialized),
    'results': {
        'create_table': {
            'checkpoint': 'checkpoint-125',
            'bleu': bleu_ct, 'sqam': sqam_ct, 'tsed': tsed_ct
        },
        'serialized': {
            'checkpoint': 'checkpoint-25',
            'eval_loss': 1.4022,
            'bleu': round(bleu_s, 2),
            'sqam': round(avg_sqam_s, 4),
            'tsed': round(avg_tsed_s, 4)
        }
    },
    'winner': 'CREATE TABLE' if bleu_ct > bleu_s else 'Serialized',
    'darija_reference': {
        'create_table': {'bleu': 36.20, 'sqam': 0.483, 'tsed': 0.286},
        'serialized':   {'bleu': 32.50, 'sqam': 0.472, 'tsed': 0.220}
    }
}

with open('/content/drive/MyDrive/HindiDB/results/ex1_hinglish_complete.json', 'w') as f:
    json.dump(ex1_results, f, indent=2)

print(f"\n✅ EX1 complete — winner: {ex1_results['winner']}")
print("Saved to Drive: results/ex1_hinglish_complete.json")

EX1: Prompt Design — Hinglish (Qwen-2.5-Coder-3B)

Prompt                   BLEU     SQAM     TSED
------------------------------------------------
CREATE TABLE            20.39   0.3190   0.3080
Serialized              14.14   0.2552   0.2811
------------------------------------------------

DarijaDB reference (Qwen-2.5-Coder-3B):
CREATE TABLE            36.20    0.483    0.286
Serialized              32.50    0.472    0.220

✅ EX1 complete — winner: CREATE TABLE
Saved to Drive: results/ex1_hinglish_complete.json


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


RE EVAL of english baseline

In [ ]:
import gc, torch

try:
    del model
    gc.collect()
    torch.cuda.empty_cache()
except: pass

print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

GPU memory: 0.00 GB


In [ ]:
# Install fine-tuning dependencies
!pip install "transformers==4.43.4" -q
!pip install "peft==0.11.1" -q
!pip install "bitsandbytes==0.49.2" --no-cache-dir -q
!pip install "accelerate==0.30.0" -q
!pip install "trl==0.8.6" -q

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))
print("✅ Logged in")

✅ Logged in


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_name = "Qwen/Qwen2.5-Coder-3B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/139 [00:00<?, ?B/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
GPU memory: 3.40 GB


In [ ]:
import json
from datasets import Dataset

# Load ENGLISH finetuning data (CREATE TABLE format, English questions)
english_dir = '/content/drive/MyDrive/HindiDB/data/finetuning_ready'

def load_as_hf_dataset(path):
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    formatted = []
    for row in data:
        full_text = row['prompt'] + row['completion'] + tokenizer.eos_token
        formatted.append({
            'text': full_text,
            'prompt': row['prompt'],
            'completion': row['completion']
        })
    return Dataset.from_list(formatted)

train_en = load_as_hf_dataset(f'{english_dir}/train.json')
dev_en   = load_as_hf_dataset(f'{english_dir}/dev.json')

print(f"English train: {len(train_en)} | Dev: {len(dev_en)}")

English train: 5232 | Dev: 528


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
import os

ckpt_dir = '/content/drive/MyDrive/HindiDB/models/qwen_english_baseline'
os.makedirs(ckpt_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=ckpt_dir,
    max_steps=400,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=16,
    load_best_model_at_end=False,
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_en,
    eval_dataset=dev_en,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=1024,
    packing=False,
)

print("✅ Trainer ready — English baseline (saving to Drive)")
print("Watch eval_loss — stop manually when it starts consistently rising")

Map:   0%|          | 0/5232 [00:00<?, ? examples/s]

Map:   0%|          | 0/528 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs


✅ Trainer ready — English baseline (saving to Drive)
Watch eval_loss — stop manually when it starts consistently rising


In [ ]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
25,0.735300,0.747512
50,0.272900,0.987109
75,0.206200,1.060791
100,0.183400,1.135427
125,0.158000,1.177871
150,0.178400,1.280904
175,0.171000,1.241196
200,0.150400,1.293034


We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an excepti

KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import gc, torch

try:
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()
except: pass

print(f"GPU memory after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")

GPU memory after cleanup: 0.00 GB


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import os, json, time

# Confirm checkpoint-175 exists
ckpt_dir = '/content/drive/MyDrive/HindiDB/models/qwen_english_baseline'
checkpoints = sorted([c for c in os.listdir(ckpt_dir) if c.startswith('checkpoint')])
print("Available checkpoints:", checkpoints)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Available checkpoints: ['checkpoint-100', 'checkpoint-125', 'checkpoint-150', 'checkpoint-175', 'checkpoint-200', 'checkpoint-25', 'checkpoint-50', 'checkpoint-75']


In [ ]:
!pip install "transformers==4.43.4" -q
!pip install "peft==0.11.1" -q
!pip install "bitsandbytes==0.49.2" --no-cache-dir -q

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-3B", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-Coder-3B",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = PeftModel.from_pretrained(
    base_model,
    f'{ckpt_dir}/checkpoint-175'
)
model.eval()
print(f"✅ Loaded checkpoint-175 (eval_loss: 1.2419)")
#print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

RuntimeError: No GPU found. A GPU is needed for quantization.